# OFV-kontroller - samlet Databricks-notebook

Python/Databricks-port av `OFV_RefreshInfo.bas`. Kjorer de tre
kontrollene og samler input og alle resultater i EN Excel-arbeidsbok
(+ en interaktiv HTML-rapport), i stedet for tre separate VBA-ark i samme
arbeidsbok.

## Hva kontrollene gjor
- **Kontroll solgte biler**: for hver bil hentes hele OFV-
  transaksjonshistorikken. Den transaksjonen som ligger **naermest**
  bokfort dato (for eller etter, likegyldig) brukes som
  kontrollgrunnlag. Har OFV ingen transaksjoner for bilen i det hele
  tatt, brukes forstegangsregistreringsdato fra Statens vegvesen
  (SVV) i stedet. Er "OrgNrSelger" fylt ut i input, flagges det
  ekstra ("Selvhandel") hvis samme selskap star som bade selger og
  kjoper i den matchede transaksjonen.
- **Varekjop bruktbil**: tre lister - Innkjop (i perioden), IB (kjopt
  forrige periode) og UB (fortsatt pa lager). For hver bil i Innkjop
  og IB beregnes "lagerperioden" (siste kjop av oppgitt
  juridisk enhet, og forste videresalg etter det kjopet) - OK sa
  lenge lagerperioden overlapper kontrollperioden. I tillegg hentes
  full historikk for biler OFV sier er kjopt av selskapet i perioden,
  men som IKKE star i Innkjop-listen ("mangler i bokforing").
- **Kontroll Demobil**: for hver bil sjekkes om bilens NYESTE
  registrerte OFV-transaksjon fortsatt har oppgitt juridisk enhet som
  kjoper (OK), eller om bilen er videreselgt (Avvik).

## Mappestruktur per selskap
Selskapsnavn settes via widgeten `SELSKAP` og styrer alle stier:
- `RåData/<Selskap>/` - ra kildefiler
- `Input/<Selskap>/` - ferdig formaterte inputfiler
- `Output/<Selskap>/` - resultater (Excel + HTML)

## Input - tre separate Excel-filer
Hver kontroll har sin EGEN inputfil, med faste filnavn i
`Input/<Selskap>/` (se Celle 1). Hver fil har to faner:

- **"Konfig"**: to kolonner, `Felt` og `Verdi` - en rad per
  innstilling (orgnr/datoer).
- **"Biler"**: en rad per bil/kjoretoy, med kolonnen `Identifikator`
  (regnr ELLER VIN - bare en av dem per rad, koden kjenner dem
  automatisk fra hverandre pa lengde: VIN er alltid 17 tegn) pluss en
  kontroll-spesifikk kolonne.

| Fil | Konfig-felt | Biler-kolonner |
|---|---|---|
| `kontroll_solgte_biler_input.xlsx` | `OrgNrSelger` (valgfri) | `Identifikator`, `BokfortDato` |
| `kontroll_varekjop_bruktbil_input.xlsx` | `OrgNrKjoper`, `DatoFra`, `DatoTil` | `Identifikator`, `Liste` (`Innkjop`/`IB`/`UB`) |
| `kontroll_demobil_input.xlsx` | `OrgNr`, `DatoFra` (valgfri), `DatoTil` (valgfri) | `Identifikator`, `BokfortInnDato` |

Datoer kan skrives som ekte Excel-datoer, som tekst `dd.mm.aaaa` eller
som ISO-tekst `aaaa-mm-dd`. Mangler en inputfil (01_Verifisering_Input
fant ingen rafil for kontrollen), hoppes den kontrollen over.

## Output
EN samlet arbeidsbok `OFV_Kontroller_<tidsstempel>.xlsx` i
`OUTPUT_DIR`, med fanene:

1. **Input** - oversikt over hvilken kildefil/konfigurasjon/liste med
   biler som ble brukt for hver av de tre kontrollene.
2. **Resultat solgte biler** / **Kontroll solgte biler**
3. **Resultat Varekjop** / **Kontroll Varekjop Bruktbil**
4. **Resultat Demobil** / **Kontroll Demobil**

I tillegg en selvstendig, interaktiv HTML-rapport
(`OFV_Kontroller_<tidsstempel>.html`, virker uten nett) med fanene
**Oversikt**, **Solgte biler**, **Varekjop bruktbil**, **Demobil** og
**Analyse** (nokkeltall og grafer). Alle tabeller kan sorteres, sokes i
og filtreres pa hver kolonne (tekst, verdiliste, min/maks, fra/til-dato
og status), og filtrert utvalg kan lastes ned som CSV.

Fargene er BDOs egen profilpalett (se Celle 1) i stedet for Excels
innebygde gront/gult/rodt - BDO bla er forbeholdt logoen og brukes
derfor ikke her.

## Forbedringer i forhold til VBA-versjonen
- JSON bygges/parses med `json`/`requests` i stedet for en
  handrullet tegn-for-tegn-parser.
- Retry bruker ekte eksponentiell backoff (+ jitter) i stedet for
  VBA sin lineaere "ventetid * forsoksnummer".
- SelgerType/KjoperType regnes direkte i Python (samme regel som
  Excel-formelen i VBA-versjonen), ikke som en live Excel-formel.
- API-nokler hentes fra Databricks secrets, aldri hardkodet.
- Samme regnr/VIN slas bare opp en gang per kjoring (hurtigbuffer) -
  resultatet er identisk, men med farre API-kall.
- Orgnr normaliseres (`normalize_orgnr`) for de sammenlignes, sa
  `933749312.0` fra Excel matcher `933749312` fra OFV.

In [ ]:
%pip install openpyxl -q

In [ ]:
# ==================================================================
# CELLE 1: KONFIGURASJON
# ==================================================================

import html
import json
import os
import random
import re
import time
from collections import Counter
from datetime import date, datetime, timedelta

import pandas as pd
import requests
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------------
# Selskapsnavn - styrer alle undermapper
# ------------------------------------------------------------------
dbutils.widgets.text("SELSKAP", "", "Selskapsnavn (mappenavn)")
SELSKAP = dbutils.widgets.get("SELSKAP").strip()
if not SELSKAP:
    raise ValueError("SELSKAP-widgeten ma fylles ut (f.eks. 'Toyota Bilia Oslo AS').")

print(f"Selskap: {SELSKAP}")

# ------------------------------------------------------------------
# Workspace-stier - per selskap
# ------------------------------------------------------------------
_BASE = "/Workspace/Users/stefan.luidold@bdo.no/OFV API"

INPUT_DIR = os.path.join(_BASE, "Input", SELSKAP)
OUTPUT_DIR = os.path.join(_BASE, "Output", SELSKAP)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")

INPUT_FILE_SOLGTE_BILER = os.path.join(INPUT_DIR, "kontroll_solgte_biler_input.xlsx")
INPUT_FILE_VAREKJOP = os.path.join(INPUT_DIR, "kontroll_varekjop_bruktbil_input.xlsx")
INPUT_FILE_DEMOBIL = os.path.join(INPUT_DIR, "kontroll_demobil_input.xlsx")

# Hvilke kontroller som kjores styres av hvilke inputfiler som finnes -
# 01_Verifisering_Input hopper over kontroller uten matchende rafil, og
# da hoppes de over her ogsa (i stedet for at hele notebooken krasjer).
KJOR_KONTROLL = {
    "solgte_biler": os.path.exists(INPUT_FILE_SOLGTE_BILER),
    "varekjop_bruktbil": os.path.exists(INPUT_FILE_VAREKJOP),
    "demobil": os.path.exists(INPUT_FILE_DEMOBIL),
}

print("\nInputfiler:")
for _navn, _sti in [("solgte_biler", INPUT_FILE_SOLGTE_BILER),
                    ("varekjop_bruktbil", INPUT_FILE_VAREKJOP),
                    ("demobil", INPUT_FILE_DEMOBIL)]:
    print(f"  {'OK     ' if KJOR_KONTROLL[_navn] else 'MANGLER'}  {_navn}: {os.path.basename(_sti)}")

if not any(KJOR_KONTROLL.values()):
    raise FileNotFoundError(
        f"Fant ingen inputfiler i {INPUT_DIR}. Kjor 01_Verifisering_Input "
        f"for selskapet '{SELSKAP}' forst."
    )

_TIDSSTEMPEL = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, f"OFV_Kontroller_{_TIDSSTEMPEL}.xlsx")
OUTPUT_HTML = os.path.join(OUTPUT_DIR, f"OFV_Kontroller_{_TIDSSTEMPEL}.html")

# ------------------------------------------------------------------
# API-endepunkter og retry-oppsett
# ------------------------------------------------------------------

OFV_URL = "https://api.ofv.no/transactions/v1/"
SVV_URL = "https://akfell-datautlevering.atlas.vegvesen.no/enkeltoppslag/kjoretoydata"

MAX_RETRIES = 4
RETRY_BASE_S = 2.0       # eksponentiell backoff: ca 2, 4, 8, 16 sek (+ jitter)
RETRY_JITTER_S = 0.5
API_PAUSE_S = 0.15       # pause mellom hvert kall, for a vaere en god API-borger
HTTP_TIMEOUT_S = 30

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

# ------------------------------------------------------------------
# Secrets - API-nokler hentes ALDRI hardkodet
# ------------------------------------------------------------------

SECRET_SCOPE = "stefan-api-keys"

try:
    OFV_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="ofv-api-key")
except Exception as exc:
    raise RuntimeError(
        f"Fant ikke OFV API-nokkel i secret scope '{SECRET_SCOPE}' "
        f"(key 'ofv-api-key'). Opprett secreten med "
        f"'databricks secrets put --scope {SECRET_SCOPE} --key ofv-api-key'. "
        f"({exc})"
    )

try:
    SVV_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="vegvesen-api-key")
except Exception:
    # SVV er kun en reserve for forstegangsregistrering (Kontroll solgte
    # biler) nar OFV ikke har noen transaksjoner for kjoretoyet - mangler
    # nokkelen, hoppes SVV-oppslaget bare over, akkurat som i VBA.
    SVV_API_KEY = None

print(f"\nSVV-reserve: {'aktiv' if SVV_API_KEY else 'av (mangler vegvesen-api-key)'}")

# ------------------------------------------------------------------
# BDOs profilpalett (se cloud.brandmaster.com/point/no/bdobc) - BDO
# Bla (#22409A) er forbeholdt logoen og brukes IKKE i rapporten.
# ------------------------------------------------------------------

BDO = {
    "red": "E81A3B",
    "charcoal": "333333",
    "slate": "5B6E7F",
    "burgundy": "98002E",
    "pale_charcoal": "E7E7E7",
    "jade": "009966",
    "ocean": "008FD2",
    "gold": "D67900",
    "white": "FFFFFF",
}

# Semantisk bruk - samme roller som fargekodene i VBA-versjonen
FARGE_HEADER_BG = BDO["charcoal"]
FARGE_HEADER_FONT = BDO["white"]
FARGE_SEKSJON_BG = BDO["pale_charcoal"]
FARGE_SEKSJON_FONT = BDO["charcoal"]
FARGE_OK_BG = BDO["jade"]
FARGE_OK_FONT = BDO["white"]
FARGE_AVVIK_BG = BDO["red"]
FARGE_AVVIK_FONT = BDO["white"]
FARGE_VARSEL_BG = BDO["gold"]
FARGE_VARSEL_FONT = BDO["white"]
FARGE_GRUPPE_BG = "F4F4F4"
FARGE_KANT = BDO["slate"]

# Grenser for fargekoding av "Dager avvik" i Kontroll solgte biler:
AVVIK_GRONN_MAX = 2
AVVIK_GUL_MAX = 14

# Felt som regnes som "bil-nivaa"
CAR_LEVEL_FIELDS = {
    "ChassisNumber", "MakeName", "ModelName", "FuelGroup",
    "IsLeased", "IsUsedImported", "FirstRegistrationDate",
}

# 24-kolonners feltkart for Resultat-tabellene
FIELD_MAP = [
    ("Input", "Input", False),
    ("Kilde", "Kilde", False),
    ("RegNo", "RegNo", False),
    ("ChassisNumber", "Chassisnummer", False),
    ("MakeName", "Merke", False),
    ("ModelName", "Modell", False),
    ("RegistrationType", "RegistreringsType", False),
    ("FuelGroup", "Drivstoffgruppe", False),
    ("IsLeased", "Leaset", False),
    ("IsUsedImported", "Bruktimportert", False),
    ("FirstRegistrationDate", "ForstegangsRegistrering", True),
    ("TransactionNumber", "TransaksjonsNummer", False),
    ("TransactionDate", "Eierskiftedato", True),
    ("SelgerType", "SelgerType", False),
    ("KjoperType", "KjoperType", False),
    ("FromOwnerType", "SelgerEierType", False),
    ("FromOwnerCompanyName", "SelgerEierFirma", False),
    ("FromOwnerOrgNo", "SelgerOrgNr", False),
    ("FromOwnerCounty", "SelgerEierFylke", False),
    ("ToOwnerType", "KjoperEierType", False),
    ("ToOwnerCompanyName", "KjoperEierFirma", False),
    ("ToOwnerOrgNo", "KjoperOrgNr", False),
    ("ToOwnerCounty", "KjoperEierFylke", False),
    ("Status", "Status", False),
]

In [ ]:
# ==================================================================
# CELLE 2: HJELPEFUNKSJONER
# ==================================================================

# ------------------------------------------------------------------
# Tekst, dato og identifikatorer
# ------------------------------------------------------------------


def _er_tom(value) -> bool:
    """None, NaN/NaT og tomme strenger regnes som tomme."""
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass
    return isinstance(value, str) and not value.strip()


def normalize_identifier(value) -> str:
    if _er_tom(value):
        return ""
    return str(value).strip().upper().replace(" ", "")


def normalize_orgnr(value) -> str:
    """Organisasjonsnummer som ren sifferstreng.

    pandas leser Konfig-kolonnen "Verdi" som float nar de andre cellene i
    kolonnen er tomme (f.eks. Demobil uten DatoFra/DatoTil), slik at
    933749312 blir 933749312.0. Uten denne normaliseringen ble orgnr
    sammenlignet som "933749312.0" mot OFV sitt "933749312", og ALLE
    demobiler ble feilaktig flagget som avvik. Haandterer ogsa NaN (tom
    celle -> ""), mellomrom ("933 749 312") og "NO ... MVA"."""
    if _er_tom(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0+", text):
        text = text.split(".")[0]
    return re.sub(r"\D", "", text)


def is_vin(text: str) -> bool:
    """VIN (chassisnummer) er alltid noyaktig 17 tegn per ISO 3779."""
    return len(text) == 17


def split_identifikator(raw) -> tuple:
    """Deler en raw Identifikator-celle inn i (regno, vin)."""
    verdi = normalize_identifier(raw)
    if not verdi:
        return "", ""
    if is_vin(verdi):
        return "", verdi
    return verdi, ""


def build_vehicle_key(vin: str, regno: str) -> str:
    if vin:
        return f"VIN|{vin}"
    if regno:
        return f"REG|{regno}"
    return ""


def parse_bokfort_dato(raw):
    """Tolker en dato skrevet inn av en bruker (Excel-celle)."""
    if _er_tom(raw):
        return None
    if isinstance(raw, pd.Timestamp):
        return raw.date()
    if isinstance(raw, datetime):
        return raw.date()
    if isinstance(raw, date):
        return raw

    text = str(raw).strip()
    if not text:
        return None

    # ISO-format (aaaa-mm-dd), f.eks. fra en tekstcelle eller widget
    iso = re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})(?:[ T].*)?", text)
    if iso:
        try:
            return date(int(iso.group(1)), int(iso.group(2)), int(iso.group(3)))
        except ValueError:
            return None

    text = text.replace("/", ".").replace("-", ".")
    parts = text.split(".")
    if len(parts) != 3:
        return None

    try:
        dag, maned, ar = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None

    if ar < 100:
        ar += 2000

    if not (1 <= maned <= 12 and 1 <= dag <= 31 and 1900 <= ar <= 2100):
        return None

    try:
        return date(ar, maned, dag)
    except ValueError:
        return None


def date_from_iso(iso_value):
    """Parser OFV/SVV sine ISO-formatterte datoer."""
    if not iso_value:
        return None
    s = str(iso_value)
    if len(s) < 10:
        return None
    try:
        return date(int(s[0:4]), int(s[5:7]), int(s[8:10]))
    except ValueError:
        return None


def find_any(obj, key):
    """Sok rekursivt etter forste forekomst av `key` i et nestet dict/list-tre."""
    if isinstance(obj, dict):
        if key in obj and obj[key] not in (None, ""):
            return obj[key]
        for v in obj.values():
            found = find_any(v, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_any(item, key)
            if found is not None:
                return found
    return None


def compute_owner_label(owner_type, company_name) -> str:
    owner_type = owner_type or ""
    company_name = company_name or ""
    if not owner_type and not company_name:
        return ""
    if owner_type == "Privat":
        return "Privat"
    return company_name or owner_type


# ------------------------------------------------------------------
# HTTP - OFV og SVV, med eksponentiell backoff
# ------------------------------------------------------------------

_SESSION = requests.Session()


def _backoff_sleep(attempt: int) -> None:
    wait = RETRY_BASE_S * (2 ** (attempt - 1)) + random.uniform(0, RETRY_JITTER_S)
    time.sleep(wait)


def post_ofv_with_retries(api_key: str, body: dict):
    """POST mot OFV med eksponentiell backoff-retry pa 429/5xx."""
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        API_TELLER["ofv_http"] += 1
        try:
            resp = _SESSION.post(
                OFV_URL,
                headers={
                    "Content-Type": "application/json",
                    "Cache-Control": "no-cache",
                    "Ocp-Apim-Subscription-Key": api_key,
                },
                json=body,
                timeout=HTTP_TIMEOUT_S,
            )
        except requests.RequestException as exc:
            last_error = str(exc)
            _backoff_sleep(attempt)
            continue

        if resp.status_code == 200:
            return resp.json(), "OK"
        if resp.status_code == 401:
            return None, "Feil: OFV 401 - kontroller API-nokkelen"
        if resp.status_code == 403:
            return None, "Feil: OFV 403 - tilgang eller kvote"
        if resp.status_code == 404:
            return None, "Feil: OFV-endepunktet ble ikke funnet"
        if resp.status_code in RETRYABLE_STATUS_CODES:
            last_error = f"{resp.status_code}: {resp.text[:200]}"
            _backoff_sleep(attempt)
            continue
        return None, f"Feil: OFV HTTP {resp.status_code}"
    return None, f"Feil: OFV - {last_error}"


def get_svv_response(api_key: str, filter_name: str, identifier: str):
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        API_TELLER["svv_http"] += 1
        try:
            resp = _SESSION.get(
                SVV_URL,
                params={filter_name: identifier},
                headers={
                    "SVV-Authorization": f"Apikey {api_key}",
                    "Accept": "application/json",
                },
                timeout=HTTP_TIMEOUT_S,
            )
        except requests.RequestException as exc:
            last_error = str(exc)
            _backoff_sleep(attempt)
            continue

        if resp.status_code == 200:
            return resp.json(), "Statens vegvesen"
        if resp.status_code == 401:
            return None, "Feil: Vegvesenet 401 - kontroller API-nokkel"
        if resp.status_code == 403:
            return None, "Feil: Vegvesenet 403 - tilgang eller kvote"
        if resp.status_code == 404:
            return None, "Statens vegvesen - ingen treff"
        if resp.status_code in RETRYABLE_STATUS_CODES:
            last_error = f"{resp.status_code}: {resp.text[:200]}"
            _backoff_sleep(attempt)
            continue
        return None, f"Feil: Vegvesenet HTTP {resp.status_code}"
    return None, f"Feil: Vegvesenet - {last_error}"


def fetch_vehicle_info_from_svv(api_key: str, regno: str, vin: str) -> dict:
    """Kalles kun nar OFV ikke har noen transaksjoner for kjoretoyet.
    Buffres per kjoretoy pa samme mate som OFV-oppslagene."""
    key = (regno, vin)
    if key in _SVV_CACHE:
        return dict(_SVV_CACHE[key])
    result = _fetch_vehicle_info_from_svv_uncached(api_key, regno, vin)
    if not str(result.get("Status") or "").startswith("Feil:"):
        _SVV_CACHE[key] = dict(result)
    return result


def _fetch_vehicle_info_from_svv_uncached(api_key: str, regno: str, vin: str) -> dict:
    result = {"Status": "Statens vegvesen - ingen dato"}
    data = None
    status = None
    if regno:
        data, status = get_svv_response(api_key, "kjennemerke", regno)
    if data is None and vin:
        data, status = get_svv_response(api_key, "understellsnummer", vin)
    if data is None:
        result["Status"] = status or "Statens vegvesen - ingen dato"
        return result
    iso_date = (
        find_any(data, "registrertForstegangNorgeDato")
        or find_any(data, "registrertForstegangDato")
    )
    d = date_from_iso(iso_date) if iso_date else None
    if d:
        result["FirstRegistrationDate"] = d
        result["Status"] = "Statens vegvesen"
    return result


# ------------------------------------------------------------------
# OFV-transaksjoner - bygging av rader + de to API-kallene
# ------------------------------------------------------------------


def build_transaction_row(identifier, tx: dict, use_vin: bool,
                           original_regno: str, original_vin: str) -> dict:
    from_side = tx.get("from") or {}
    to_side = tx.get("to") or {}
    from_owner = from_side.get("owner") or {}
    to_owner = to_side.get("owner") or {}
    from_company = from_owner.get("companyInfo") or {}
    to_company = to_owner.get("companyInfo") or {}
    from_type = from_owner.get("type") or ""
    from_company_name = from_company.get("name") or ""
    to_type = to_owner.get("type") or ""
    to_company_name = to_company.get("name") or ""

    return {
        "Input": identifier,
        "Kilde": "VIN" if use_vin else "Regnr",
        "RegNo": tx.get("regNo") or original_regno,
        "ChassisNumber": tx.get("chassisNumber") or original_vin,
        "MakeName": tx.get("makeName"),
        "ModelName": tx.get("modelName"),
        "RegistrationType": tx.get("registrationType"),
        "FuelGroup": tx.get("fuelGroup"),
        "IsLeased": tx.get("isLeased"),
        "IsUsedImported": tx.get("isUsedImported"),
        "FirstRegistrationDate": date_from_iso(tx.get("firstRegistrationDate")),
        "TransactionNumber": tx.get("transactionNumber"),
        "TransactionDate": date_from_iso(tx.get("transactionDate")),
        "SelgerType": compute_owner_label(from_type, from_company_name),
        "KjoperType": compute_owner_label(to_type, to_company_name),
        "FromOwnerType": from_type,
        "FromOwnerCompanyName": from_company_name,
        "FromOwnerOrgNo": from_company.get("organizationNumber"),
        "FromOwnerCounty": from_owner.get("countyName"),
        "ToOwnerType": to_type,
        "ToOwnerCompanyName": to_company_name,
        "ToOwnerOrgNo": to_company.get("organizationNumber"),
        "ToOwnerCounty": to_owner.get("countyName"),
        "Status": "OK",
    }


def build_empty_transaction_row(identifier, use_vin: bool,
                                 original_regno: str, original_vin: str,
                                 status_text: str) -> dict:
    return {
        "Input": identifier,
        "Kilde": "VIN" if use_vin else "Regnr",
        "RegNo": original_regno,
        "ChassisNumber": original_vin,
        "Status": status_text,
    }


# Hurtigbuffer per kjoring: samme regnr/VIN star ofte flere ganger i
# input (f.eks. en bil som er solgt flere ganger), og OFV-historikken er
# den samme hver gang. Buffret tommes ved starten av hver kjoring (Celle
# 3), og feilede kall buffres aldri - de forsokes pa nytt neste gang.
_OFV_CACHE = {}
_SVV_CACHE = {}
API_TELLER = Counter()


def fetch_ofv_transactions(api_key: str, identifier: str, use_vin: bool,
                            original_regno: str, original_vin: str) -> list:
    """Som _fetch_ofv_transactions_uncached, men med hurtigbuffer per
    kjoretoy. Returnerer en NY liste med KOPIER av radene hver gang, sa
    resultatet er identisk med et ferskt API-kall."""
    key = ("VIN" if use_vin else "REG", normalize_identifier(identifier),
           original_regno, original_vin)
    if key in _OFV_CACHE:
        API_TELLER["ofv_buffer"] += 1
        return [dict(r) for r in _OFV_CACHE[key]]
    rows = _fetch_ofv_transactions_uncached(
        api_key, identifier, use_vin, original_regno, original_vin)
    API_TELLER["ofv_kjoretoy"] += 1
    if not any(str(r.get("Status") or "").startswith("Feil:") for r in rows):
        _OFV_CACHE[key] = [dict(r) for r in rows]
    return rows


def _fetch_ofv_transactions_uncached(api_key: str, identifier: str, use_vin: bool,
                                      original_regno: str, original_vin: str) -> list:
    """Henter ALLE transaksjoner for kjoretoyet - ett kall per kjoretoy."""
    filter_key = "chassisNumber" if use_vin else "regNo"
    rows = []
    cursor = None
    while True:
        pagination = {"first": 1000}
        if cursor:
            pagination["cursor"] = cursor
        body = {
            "filters": {filter_key: identifier},
            "pagination": pagination,
            "sorting": {"orderBy": "transactionDate", "orderDirection": "DESC"},
        }
        data, status = post_ofv_with_retries(api_key, body)
        if status != "OK":
            rows.append(build_empty_transaction_row(
                identifier, use_vin, original_regno, original_vin, status))
            return rows
        for tx in (data or {}).get("transactions", []):
            rows.append(build_transaction_row(
                identifier, tx, use_vin, original_regno, original_vin))
        pg = (data or {}).get("pagination", {})
        if pg.get("hasNextPage"):
            cursor = pg.get("endCursor")
            if not cursor:
                break
            time.sleep(API_PAUSE_S)
        else:
            break
    if not rows:
        rows.append(build_empty_transaction_row(
            identifier, use_vin, original_regno, original_vin,
            "Ingen eierskifter funnet"))
    return rows


def fetch_ofv_transactions_by_buyer_org(api_key: str, org_no: str,
                                         date_fra: date, date_til: date) -> list:
    """Henter ALLE OFV-transaksjoner der gitt orgnr star som KJOPER."""
    rows = []
    cursor = None
    while True:
        pagination = {"first": 1000}
        if cursor:
            pagination["cursor"] = cursor
        body = {
            "filters": {
                "toOrganizationNumber": org_no,
                "transactionDateFrom": date_fra.strftime("%Y-%m-%d"),
                "transactionDateTo": date_til.strftime("%Y-%m-%d"),
            },
            "pagination": pagination,
            "sorting": {"orderBy": "transactionDate", "orderDirection": "DESC"},
        }
        data, status = post_ofv_with_retries(api_key, body)
        if status != "OK":
            raise RuntimeError(f"FetchOFVTransactionsByBuyerOrg: {status}")
        for tx in (data or {}).get("transactions", []):
            rows.append(build_transaction_row("", tx, False, "", ""))
        pg = (data or {}).get("pagination", {})
        if pg.get("hasNextPage"):
            cursor = pg.get("endCursor")
            if not cursor:
                break
            time.sleep(API_PAUSE_S)
        else:
            break
    return rows


# ------------------------------------------------------------------
# Input-lesere
# ------------------------------------------------------------------


def read_konfig(path: str) -> dict:
    df = pd.read_excel(path, sheet_name="Konfig")
    konfig = {}
    for _, row in df.iterrows():
        felt = str(row.get("Felt", "")).strip()
        if felt and felt.lower() != "nan":
            verdi = row.get("Verdi")
            konfig[felt] = None if _er_tom(verdi) else verdi
    return konfig


def read_biler(path: str, extra_col: str) -> list:
    df = pd.read_excel(path, sheet_name="Biler")
    out = []
    for _, row in df.iterrows():
        raw_ident = row.get("Identifikator")
        if raw_ident is None or (isinstance(raw_ident, float) and pd.isna(raw_ident)):
            continue
        regno, vin = split_identifikator(raw_ident)
        if not regno and not vin:
            continue
        out.append({
            "regno": regno,
            "vin": vin,
            "identifikator": vin or regno,
            extra_col: row.get(extra_col) if extra_col in df.columns else None,
        })
    return out


# ------------------------------------------------------------------
# Resultat-tabell - delt av alle tre kontroller
# ------------------------------------------------------------------


def build_resultat_rows(all_rows: list) -> list:
    records = []
    forrige_input = None
    for row in all_rows:
        denne_input = row.get("Input")
        er_forste = (denne_input != forrige_input)
        forrige_input = denne_input
        record = {"_ErForsteIGruppe": er_forste, "_Input": denne_input}
        for key, header, _is_date in FIELD_MAP:
            if key in ("SelgerType", "KjoperType"):
                record[header] = row.get(key)
            elif not er_forste and key in CAR_LEVEL_FIELDS:
                record[header] = None
            else:
                record[header] = row.get(key)
        records.append(record)
    return records


# ------------------------------------------------------------------
# Excel-stiler (BDO-palett)
# ------------------------------------------------------------------

_THIN_BORDER_SIDE = Side(style="thin", color=FARGE_KANT)
THIN_BORDER = Border(
    left=_THIN_BORDER_SIDE, right=_THIN_BORDER_SIDE,
    top=_THIN_BORDER_SIDE, bottom=_THIN_BORDER_SIDE,
)


def set_title_bar(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(bold=True, size=14, color=FARGE_HEADER_FONT)
    cell.fill = PatternFill("solid", fgColor=FARGE_HEADER_BG)
    cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 26


def set_explanation_row(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(italic=True)
    cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.row_dimensions[row].height = 30


def set_section_bar(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(bold=True, size=12, color=FARGE_SEKSJON_FONT)
    cell.fill = PatternFill("solid", fgColor=FARGE_SEKSJON_BG)
    cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20


def set_kpi_box(ws, row: int, col_from: int, col_to: int, label: str,
                 value, bg_hex: str = None, font_hex: str = None,
                 big: bool = False) -> None:
    ws.merge_cells(start_row=row, start_column=col_from, end_row=row, end_column=col_to)
    cell = ws.cell(row=row, column=col_from, value=label)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    if bg_hex:
        cell.fill = PatternFill("solid", fgColor=bg_hex)
        cell.font = Font(bold=True, color=font_hex or "000000")
    ws.merge_cells(start_row=row + 1, start_column=col_from, end_row=row + 1, end_column=col_to)
    vcell = ws.cell(row=row + 1, column=col_from, value=value)
    vcell.font = Font(bold=True, size=16 if big else 12, color=font_hex or "000000")
    vcell.alignment = Alignment(horizontal="center", vertical="center")
    if bg_hex:
        vcell.fill = PatternFill("solid", fgColor=bg_hex)


def style_table_header(ws, row: int, headers: list) -> None:
    for i, header in enumerate(headers, start=1):
        cell = ws.cell(row=row, column=i, value=header)
        cell.font = Font(bold=True, color=FARGE_HEADER_FONT)
        cell.fill = PatternFill("solid", fgColor=FARGE_HEADER_BG)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 30


def color_status_cell(cell, status_text: str) -> None:
    status_text = status_text or ""
    if status_text.startswith("OK"):
        cell.fill = PatternFill("solid", fgColor=FARGE_OK_BG)
        cell.font = Font(color=FARGE_OK_FONT, bold=True)
    elif status_text.startswith("Avvik"):
        cell.fill = PatternFill("solid", fgColor=FARGE_AVVIK_BG)
        cell.font = Font(color=FARGE_AVVIK_FONT, bold=True)


def set_column_widths(ws, widths: dict) -> None:
    for col_letter, width in widths.items():
        ws.column_dimensions[col_letter].width = width


def set_date_format(ws, col_letter: str, first_row: int, last_row: int) -> None:
    if last_row < first_row:
        return
    for row in range(first_row, last_row + 1):
        ws[f"{col_letter}{row}"].number_format = "dd.mm.yyyy"

In [ ]:
# ==================================================================
# CELLE 3: HOVEDLOGIKK
# ==================================================================

# ------------------------------------------------------------------
# Kontroll 1: Kontroll solgte biler
# ------------------------------------------------------------------


def build_kontroll_row(regno: str, vin: str, bokfort_raw, tx_rows: list,
                        svv_info, seller_org_no: str) -> dict:
    bokfort_date = parse_bokfort_dato(bokfort_raw)
    seller_norm = normalize_orgnr(seller_org_no)

    model_name = ""
    chassis_no = vin
    regno_resolved = regno
    first_reg_date = None
    has_any_ok = False
    error_status = ""

    nearest_row = None
    nearest_diff = None

    for r in tx_rows:
        status = r.get("Status")
        if status == "OK":
            has_any_ok = True
            if not model_name:
                model_name = r.get("ModelName") or ""
            if r.get("ChassisNumber"):
                chassis_no = r["ChassisNumber"]
            if r.get("RegNo"):
                regno_resolved = r["RegNo"]
            if first_reg_date is None and r.get("FirstRegistrationDate"):
                first_reg_date = r["FirstRegistrationDate"]
            tx_date = r.get("TransactionDate")
            if tx_date and bokfort_date:
                diff = abs((tx_date - bokfort_date).days)
                if nearest_diff is None or diff < nearest_diff:
                    nearest_diff = diff
                    nearest_row = r
        elif isinstance(status, str) and status.startswith("Feil:"):
            if not error_status:
                error_status = status

    if not has_any_ok and svv_info and svv_info.get("FirstRegistrationDate"):
        first_reg_date = svv_info["FirstRegistrationDate"]

    result = {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "BokfortDato": bokfort_date,
        "Forstegangsregistrert": first_reg_date,
        "KontrollTransaksjonDato": None,
        "RegistreringsType": "",
        "DagerAvvik": None,
        "Selger": "",
        "Kjoper": "",
        "Kilde": "Ingen",
        "Selvhandel": "",
    }

    if bokfort_date is None:
        if has_any_ok or first_reg_date is not None:
            result["ApiTreff"] = "Mangler bokfort dato"
        elif error_status:
            result["ApiTreff"] = error_status
        else:
            result["ApiTreff"] = "Ingen treff"
        result["Kontrollert"] = "Nei"

    elif nearest_row is not None:
        result["KontrollTransaksjonDato"] = nearest_row["TransactionDate"]
        result["RegistreringsType"] = nearest_row.get("RegistrationType") or ""
        result["DagerAvvik"] = nearest_diff
        result["ApiTreff"] = "Treff OFV eierskifte"
        result["Kontrollert"] = "Ja"
        result["Kilde"] = "OFV"
        result["Selger"] = compute_owner_label(
            nearest_row.get("FromOwnerType"), nearest_row.get("FromOwnerCompanyName"))
        result["Kjoper"] = compute_owner_label(
            nearest_row.get("ToOwnerType"), nearest_row.get("ToOwnerCompanyName"))
        if seller_norm:
            fra_org = normalize_orgnr(nearest_row.get("FromOwnerOrgNo"))
            til_org = normalize_orgnr(nearest_row.get("ToOwnerOrgNo"))
            result["Selvhandel"] = "Ja" if (
                fra_org == seller_norm and til_org == seller_norm) else "Nei"

    elif first_reg_date is not None:
        result["KontrollTransaksjonDato"] = first_reg_date
        result["RegistreringsType"] = "Forstegangsregistrering (SVV)"
        result["DagerAvvik"] = abs((first_reg_date - bokfort_date).days)
        result["ApiTreff"] = "Treff SVV forstegangsregistrering"
        result["Kontrollert"] = "Ja"
        result["Kilde"] = "SVV"

    else:
        if svv_info is None:
            result["ApiTreff"] = error_status or "Ingen treff"
        else:
            result["ApiTreff"] = svv_info.get("Status", "Ingen treff")
        result["Kontrollert"] = "Nei"

    return result


def run_kontroll_solgte_biler() -> dict:
    print("=== Kontroll solgte biler ===")
    konfig = read_konfig(INPUT_FILE_SOLGTE_BILER)
    seller_org_no = normalize_orgnr(konfig.get("OrgNrSelger"))
    biler = read_biler(INPUT_FILE_SOLGTE_BILER, "BokfortDato")

    all_rows = []
    kontroll_rows = []
    total = len(biler)
    svv_treff = 0
    svv_feil = 0
    ofv_feil = 0
    ofv_uten_treff = 0

    for i, bil in enumerate(biler, start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        use_vin = bool(vin)
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, use_vin, regno, vin)
        all_rows.extend(tx_rows)
        has_any_ok = any(r.get("Status") == "OK" for r in tx_rows)
        if not has_any_ok:
            if any(isinstance(r.get("Status"), str) and r["Status"].startswith("Feil:")
                   for r in tx_rows):
                ofv_feil += 1
            else:
                ofv_uten_treff += 1
        svv_info = None
        if not has_any_ok and SVV_API_KEY:
            print(f"  SVV: Forstegangsregistrering {ident}  ({i}/{total})")
            svv_info = fetch_vehicle_info_from_svv(SVV_API_KEY, regno, vin)
            if svv_info.get("FirstRegistrationDate"):
                svv_treff += 1
            if isinstance(svv_info.get("Status"), str) and svv_info["Status"].startswith("Feil:"):
                svv_feil += 1
        kontroll_rows.append(
            build_kontroll_row(regno, vin, bil["BokfortDato"], tx_rows, svv_info, seller_org_no))
        time.sleep(API_PAUSE_S)

    kontroll_rows_sortert = sorted(
        kontroll_rows,
        key=lambda r: r["DagerAvvik"] if isinstance(r["DagerAvvik"], (int, float)) else -1,
        reverse=True,
    )

    return {
        "kontroll": "Kontroll solgte biler",
        "kildefil": INPUT_FILE_SOLGTE_BILER,
        "orgnr": seller_org_no,
        "dato_fra": None,
        "dato_til": None,
        "antall_biler": total,
        "biler_manifest": biler,
        "all_rows": all_rows,
        "kontroll_rows": kontroll_rows_sortert,
        "stats": {
            "Antall biler": total,
            "OFV-treff": sum(1 for r in kontroll_rows if r["Kilde"] == "OFV"),
            "Uten OFV-treff": ofv_uten_treff,
            "OFV-feil": ofv_feil,
            "SVV-forstegangsregistreringer": svv_treff,
            "SVV-feil": svv_feil,
        },
    }


# ------------------------------------------------------------------
# Kontroll 2: Varekjop bruktbil
# ------------------------------------------------------------------


def finn_lagerperiode(tx_rows: list, buyer_org_no: str, kjop_grense: date):
    buyer_norm = normalize_orgnr(buyer_org_no)
    chassis_no = ""
    model_name = ""
    regno_resolved = ""
    kjopt_row = None
    solgt_row = None

    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
        tx_date = r.get("TransactionDate")
        if not tx_date:
            continue
        if (normalize_orgnr(r.get("ToOwnerOrgNo")) == buyer_norm
                and tx_date <= kjop_grense):
            if kjopt_row is None or tx_date > kjopt_row["TransactionDate"]:
                kjopt_row = r

    if kjopt_row is not None:
        for r in tx_rows:
            if r.get("Status") != "OK":
                continue
            tx_date = r.get("TransactionDate")
            if not tx_date:
                continue
            if (normalize_orgnr(r.get("FromOwnerOrgNo")) == buyer_norm
                    and tx_date > kjopt_row["TransactionDate"]):
                if solgt_row is None or tx_date < solgt_row["TransactionDate"]:
                    solgt_row = r

    return kjopt_row, solgt_row, chassis_no, model_name, regno_resolved


def lager_status_tekst(kjopt_row, solgt_row, dato_fra: date) -> str:
    if kjopt_row is None:
        return "Avvik: fant ikke opprinnelig kjop hos forhandleren"
    if solgt_row is None:
        return "OK - bilen var pa lager i perioden"
    if solgt_row["TransactionDate"] >= dato_fra:
        return "OK - bilen var pa lager i perioden"
    return "Avvik: solgt for perioden startet"


def build_innkjop_eller_ib_row(regno: str, vin: str, tx_rows: list, buyer_org_no: str,
                                dato_fra: date, dato_til: date, kjop_grense: date,
                                flagg_etikett: str, flagg_verdi: str) -> dict:
    kjopt_row, solgt_row, chassis_no, model_name, regno_resolved = finn_lagerperiode(
        tx_rows, buyer_org_no, kjop_grense)
    result = {
        "RegnrInput": regno_resolved or regno,
        "Chassisnummer": chassis_no or vin,
        "Modell": model_name,
        "FlaggEtikett": flagg_etikett,
        "FlaggVerdi": flagg_verdi,
        "FraDato": kjopt_row["TransactionDate"] if kjopt_row else None,
        "KjoptFra": compute_owner_label(
            kjopt_row.get("FromOwnerType"), kjopt_row.get("FromOwnerCompanyName")
        ) if kjopt_row else "",
        "TilDato": solgt_row["TransactionDate"] if solgt_row else None,
        "Kjoper": compute_owner_label(
            solgt_row.get("ToOwnerType"), solgt_row.get("ToOwnerCompanyName")
        ) if solgt_row else "",
    }
    result["Status"] = lager_status_tekst(kjopt_row, solgt_row, dato_fra)
    return result


def build_manglende_bokforing_row(ofv_row: dict, tx_rows: list, buyer_org_no: str,
                                   dato_til: date) -> dict:
    kjopt_row, solgt_row, chassis_no, model_name, regno_resolved = finn_lagerperiode(
        tx_rows, buyer_org_no, dato_til)
    if kjopt_row is None:
        kjopt_row = ofv_row
    result = {
        "RegnrInput": regno_resolved or ofv_row.get("RegNo") or "",
        "Chassisnummer": chassis_no or ofv_row.get("ChassisNumber") or "",
        "Modell": model_name or ofv_row.get("ModelName") or "",
        "FlaggEtikett": "I OFV-liste",
        "FlaggVerdi": "Ja",
        "FraDato": kjopt_row.get("TransactionDate"),
        "KjoptFra": compute_owner_label(
            kjopt_row.get("FromOwnerType"), kjopt_row.get("FromOwnerCompanyName")),
        "TilDato": solgt_row["TransactionDate"] if solgt_row else None,
        "Kjoper": compute_owner_label(
            solgt_row.get("ToOwnerType"), solgt_row.get("ToOwnerCompanyName")
        ) if solgt_row else "",
        "Status": "Avvik: OFV viser kjop, mangler i bokforing",
    }
    return result


def build_ub_row(regno: str, vin: str, tx_rows: list, er_i_ib: bool,
                  er_i_innkjop: bool) -> dict:
    chassis_no, model_name, regno_resolved = vin, "", regno
    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
    return {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "FunnetIIB": "Ja" if er_i_ib else "Nei",
        "FunnetIInnkjop": "Ja" if er_i_innkjop else "Nei",
        "Status": "OK - forklart" if (er_i_ib or er_i_innkjop) else "Avvik: uforklart lagerbeholdning",
    }


def run_kontroll_varekjop_bruktbil() -> dict:
    print("=== Varekjop bruktbil ===")
    konfig = read_konfig(INPUT_FILE_VAREKJOP)
    buyer_org_no = normalize_orgnr(konfig.get("OrgNrKjoper"))
    dato_fra = parse_bokfort_dato(konfig.get("DatoFra"))
    dato_til = parse_bokfort_dato(konfig.get("DatoTil"))
    if not buyer_org_no or not dato_fra or not dato_til:
        raise ValueError(
            "Varekjop bruktbil: OrgNrKjoper/DatoFra/DatoTil ma vaere fylt ut "
            "i Konfig-fanen."
        )
    biler = read_biler(INPUT_FILE_VAREKJOP, "Liste")
    queue_innkjop, queue_ib, queue_ub = {}, {}, {}
    for bil in biler:
        key = build_vehicle_key(bil["vin"], bil["regno"])
        if not key:
            continue
        liste = str(bil.get("Liste") or "").strip().lower()
        if liste == "innkjop":
            queue_innkjop[key] = bil
        elif liste == "ib":
            queue_ib[key] = bil
        elif liste == "ub":
            queue_ub[key] = bil
    print(f"  Innkjop: {len(queue_innkjop)}, IB: {len(queue_ib)}, UB: {len(queue_ub)}")

    # Steg 1: alle biler OFV sier er kjopt av orgnr i perioden
    print(f"  OFV: henter kjopsliste for org {buyer_org_no} ({dato_fra} - {dato_til}) ...")
    ofv_liste = fetch_ofv_transactions_by_buyer_org(OFV_API_KEY, buyer_org_no, dato_fra, dato_til)
    print(f"  OFV: {len(ofv_liste)} transaksjoner funnet i perioden.")
    ofv_regnr = {normalize_identifier(r["RegNo"]) for r in ofv_liste if r.get("RegNo")}
    ofv_vin = {normalize_identifier(r["ChassisNumber"]) for r in ofv_liste if r.get("ChassisNumber")}

    # Steg 2: full transaksjonshistorikk for hver bil i Innkjop + IB
    queue_api = {**queue_innkjop, **queue_ib}
    vehicle_rows_by_key = {}
    all_rows = []
    total = len(queue_api)
    for i, (key, bil) in enumerate(queue_api.items(), start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin), regno, vin)
        vehicle_rows_by_key[key] = tx_rows
        all_rows.extend(tx_rows)
        time.sleep(API_PAUSE_S)

    # Seksjon A: Innkjop i perioden
    seksjon_a = []
    for key, bil in queue_innkjop.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        er_i_ofv_liste = (
            (regno and normalize_identifier(regno) in ofv_regnr)
            or (vin and normalize_identifier(vin) in ofv_vin)
        )
        seksjon_a.append(build_innkjop_eller_ib_row(
            regno, vin, tx_rows, buyer_org_no, dato_fra, dato_til, dato_til,
            "I OFV-liste", "Ja" if er_i_ofv_liste else "Nei",
        ))

    # Biler OFV sier er kjopt i perioden, men IKKE i Innkjop-listen
    for ofv_row in ofv_liste:
        reg_n = normalize_identifier(ofv_row.get("RegNo"))
        vin_n = normalize_identifier(ofv_row.get("ChassisNumber"))
        funnet = any(
            (reg_n and reg_n == normalize_identifier(b["regno"]))
            or (vin_n and vin_n == normalize_identifier(b["vin"]))
            for b in queue_innkjop.values()
        )
        if funnet:
            continue
        key = build_vehicle_key(vin_n, reg_n)
        tx_rows = vehicle_rows_by_key.get(key)
        if tx_rows is None:
            ident = vin_n or reg_n
            print(f"  OFV: Eierskifter (mangler i bokforing) {ident}")
            tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin_n), reg_n, vin_n)
            vehicle_rows_by_key[key] = tx_rows
            all_rows.extend(tx_rows)
            time.sleep(API_PAUSE_S)
        seksjon_a.append(build_manglende_bokforing_row(ofv_row, tx_rows, buyer_org_no, dato_til))

    # Seksjon B: IB (kjopt forrige periode)
    seksjon_b = []
    kjop_grense_ib = dato_fra - timedelta(days=1)
    for key, bil in queue_ib.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        er_pa_ub = key in queue_ub
        seksjon_b.append(build_innkjop_eller_ib_row(
            regno, vin, tx_rows, buyer_org_no, dato_fra, dato_til, kjop_grense_ib,
            "Pa UB-liste", "Ja" if er_pa_ub else "Nei",
        ))

    # Seksjon C: UB (fortsatt pa lager)
    seksjon_c = []
    for key, bil in queue_ub.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        seksjon_c.append(build_ub_row(
            regno, vin, tx_rows, key in queue_ib, key in queue_innkjop))

    return {
        "kontroll": "Varekjop bruktbil",
        "kildefil": INPUT_FILE_VAREKJOP,
        "orgnr": buyer_org_no,
        "dato_fra": dato_fra,
        "dato_til": dato_til,
        "antall_biler": len(biler),
        "biler_manifest": biler,
        "all_rows": all_rows,
        "seksjon_a": seksjon_a,
        "seksjon_b": seksjon_b,
        "seksjon_c": seksjon_c,
        "stats": {
            "Innkjop (antall)": len(queue_innkjop),
            "IB (antall)": len(queue_ib),
            "UB (antall)": len(queue_ub),
            "Innkjop - OK": sum(1 for r in seksjon_a if r["Status"].startswith("OK")),
            "Innkjop - Avvik": sum(1 for r in seksjon_a if r["Status"].startswith("Avvik")
                                   and "mangler i bokforing" not in r["Status"]),
            "OFV-kjop uten bokforing": sum(
                1 for r in seksjon_a if "mangler i bokforing" in r["Status"]),
        },
    }


# ------------------------------------------------------------------
# Kontroll 3: Kontroll Demobil
# ------------------------------------------------------------------


def build_demobil_row(regno: str, vin: str, bokfort_inn_raw, tx_rows: list,
                       company_org_no: str) -> dict:
    bokfort_date = parse_bokfort_dato(bokfort_inn_raw)
    company_norm = normalize_orgnr(company_org_no)
    model_name = ""
    chassis_no = vin
    regno_resolved = regno
    nyeste_row = None
    company_navn = ""

    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
        if not company_navn and normalize_orgnr(r.get("ToOwnerOrgNo")) == company_norm:
            company_navn = r.get("ToOwnerCompanyName") or ""
        tx_date = r.get("TransactionDate")
        if tx_date and (nyeste_row is None or tx_date > nyeste_row["TransactionDate"]):
            nyeste_row = r

    result = {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "BokfortInnDato": bokfort_date,
        "SisteTransaksjonsDato": None,
        "SisteKjoper": "",
        "EidAvEnhet": "",
    }

    if nyeste_row is None:
        result["Status"] = "Ingen treff - ingen OFV-transaksjoner funnet"
    else:
        result["SisteTransaksjonsDato"] = nyeste_row["TransactionDate"]
        result["SisteKjoper"] = compute_owner_label(
            nyeste_row.get("ToOwnerType"), nyeste_row.get("ToOwnerCompanyName"))
        if normalize_orgnr(nyeste_row.get("ToOwnerOrgNo")) == company_norm:
            navn = company_navn or nyeste_row.get("ToOwnerCompanyName") or company_org_no
            result["EidAvEnhet"] = "Ja"
            result["Status"] = f"OK - fortsatt eid av {navn}"
        else:
            result["EidAvEnhet"] = "Nei"
            result["Status"] = f"Avvik - siste eierskifte er til {result['SisteKjoper']}"

    return result


def run_kontroll_demobil() -> dict:
    print("=== Kontroll Demobil ===")
    konfig = read_konfig(INPUT_FILE_DEMOBIL)
    company_org_no = normalize_orgnr(konfig.get("OrgNr"))
    dato_fra = parse_bokfort_dato(konfig.get("DatoFra"))
    dato_til = parse_bokfort_dato(konfig.get("DatoTil"))
    if not company_org_no:
        raise ValueError("Kontroll Demobil: OrgNr ma vaere fylt ut i Konfig-fanen.")
    biler = read_biler(INPUT_FILE_DEMOBIL, "BokfortInnDato")
    all_rows = []
    kontroll_rows = []
    total = len(biler)
    for i, bil in enumerate(biler, start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin), regno, vin)
        all_rows.extend(tx_rows)
        kontroll_rows.append(
            build_demobil_row(regno, vin, bil["BokfortInnDato"], tx_rows, company_org_no))
        time.sleep(API_PAUSE_S)

    return {
        "kontroll": "Kontroll Demobil",
        "kildefil": INPUT_FILE_DEMOBIL,
        "orgnr": company_org_no,
        "dato_fra": dato_fra,
        "dato_til": dato_til,
        "antall_biler": total,
        "biler_manifest": biler,
        "all_rows": all_rows,
        "kontroll_rows": kontroll_rows,
        "stats": {
            "Antall biler": total,
            "Fortsatt hos enhet": sum(
                1 for r in kontroll_rows if r["Status"].startswith("OK")),
            "Avvik (videresolgt)": sum(
                1 for r in kontroll_rows if r["Status"].startswith("Avvik")),
            "Ingen OFV-treff": sum(
                1 for r in kontroll_rows if r["Status"].startswith("Ingen treff")),
        },
    }


# ------------------------------------------------------------------
# Kjor alle tre kontrollene
# ------------------------------------------------------------------

_OFV_CACHE.clear()
_SVV_CACHE.clear()
API_TELLER.clear()

_t0 = time.time()
TIDSBRUK_PER_KONTROLL = {}


def _kjor(navn: str, funksjon):
    """Kjorer en kontroll hvis inputfilen finnes, ellers None."""
    if not KJOR_KONTROLL.get(navn):
        print(f"=== {navn}: hoppet over (inputfil mangler) ===")
        return None
    t = time.time()
    resultat = funksjon()
    TIDSBRUK_PER_KONTROLL[resultat["kontroll"]] = time.time() - t
    print(f"  Ferdig pa {time.time() - t:.1f} s\n")
    return resultat


resultat_solgte_biler = _kjor("solgte_biler", run_kontroll_solgte_biler)
resultat_varekjop = _kjor("varekjop_bruktbil", run_kontroll_varekjop_bruktbil)
resultat_demobil = _kjor("demobil", run_kontroll_demobil)

_elapsed_s = time.time() - _t0

print(f"OFV: {API_TELLER['ofv_kjoretoy']} kjoretoy slatt opp "
      f"({API_TELLER['ofv_buffer']} gjenbrukt fra buffer, "
      f"{API_TELLER['ofv_http']} HTTP-kall totalt). "
      f"SVV: {API_TELLER['svv_http']} HTTP-kall.")

In [ ]:
# ==================================================================
# CELLE 4: OUTPUT - samlet Excel-arbeidsbok
# ==================================================================

# ------------------------------------------------------------------
# Felles
# ------------------------------------------------------------------


def finn_orgnavn(all_rows: list, org_no: str) -> str:
    """Firmanavnet OFV oppgir for orgnr (forste treff som kjoper/selger)."""
    org_norm = normalize_orgnr(org_no)
    if not org_norm:
        return ""
    for side in ("ToOwner", "FromOwner"):
        for row in all_rows:
            if normalize_orgnr(row.get(f"{side}OrgNo")) == org_norm:
                navn = row.get(f"{side}CompanyName") or ""
                if navn:
                    return navn
    return ""


# ------------------------------------------------------------------
# "Input"-fane
# ------------------------------------------------------------------


def skriv_input_fane(wb: Workbook, bundles: list) -> None:
    ws = wb.create_sheet("Input")
    set_title_bar(ws, 1, 6, "Input - oversikt over kildefiler og konfigurasjon")
    ws.row_dimensions[1].height = 26
    set_section_bar(ws, 3, 6, "Konfigurasjon brukt per kontroll")
    headers = ["Kontroll", "Kildefil", "Orgnr", "Dato fra", "Dato til", "Antall biler"]
    style_table_header(ws, 4, headers)
    r = 5
    for b in bundles:
        ws.cell(row=r, column=1, value=b["kontroll"])
        ws.cell(row=r, column=2, value=b["kildefil"])
        ws.cell(row=r, column=3, value=b["orgnr"])
        if b["dato_fra"]:
            ws.cell(row=r, column=4, value=b["dato_fra"]).number_format = "dd.mm.yyyy"
        if b["dato_til"]:
            ws.cell(row=r, column=5, value=b["dato_til"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=6, value=b["antall_biler"])
        r += 1

    r += 2
    set_section_bar(ws, r, 6, "Biler lest per kontroll")
    r += 1
    headers2 = ["Kontroll", "Regnr/VIN", "Bokfort dato", "Liste-type", "Bokfort inn dato"]
    style_table_header(ws, r, headers2)
    r += 1

    for b in bundles:
        for bil in b["biler_manifest"]:
            ws.cell(row=r, column=1, value=b["kontroll"])
            ws.cell(row=r, column=2, value=bil["identifikator"])
            if "BokfortDato" in bil and bil["BokfortDato"] is not None:
                d = parse_bokfort_dato(bil["BokfortDato"])
                if d:
                    ws.cell(row=r, column=3, value=d).number_format = "dd.mm.yyyy"
            if "Liste" in bil and bil["Liste"]:
                ws.cell(row=r, column=4, value=str(bil["Liste"]))
            if "BokfortInnDato" in bil and bil["BokfortInnDato"] is not None:
                d = parse_bokfort_dato(bil["BokfortInnDato"])
                if d:
                    ws.cell(row=r, column=5, value=d).number_format = "dd.mm.yyyy"
            r += 1

    set_column_widths(ws, {"A": 26, "B": 45, "C": 16, "D": 18, "E": 16, "F": 14})


# ------------------------------------------------------------------
# "Resultat ..." - full transaksjonshistorikk
# ------------------------------------------------------------------


def skriv_resultat_fane(wb: Workbook, sheet_name: str, all_rows: list) -> None:
    ws = wb.create_sheet(sheet_name[:31])
    records = build_resultat_rows(all_rows)
    headers = [header for _key, header, _is_date in FIELD_MAP]
    style_table_header(ws, 1, headers)

    for r_idx, rec in enumerate(records, start=2):
        for c_idx, (_key, header, is_date) in enumerate(FIELD_MAP, start=1):
            value = rec.get(header)
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            if is_date and value:
                cell.number_format = "dd.mm.yyyy"
        if rec["_ErForsteIGruppe"]:
            for c_idx in range(1, len(headers) + 1):
                ws.cell(row=r_idx, column=c_idx).fill = PatternFill("solid", fgColor=FARGE_GRUPPE_BG)
                ws.cell(row=r_idx, column=c_idx).font = Font(bold=True)

    widths = {get_column_letter(i): 18 for i in range(1, len(headers) + 1)}
    widths.update({"A": 14, "D": 20, "P": 25, "Q": 25, "U": 25})
    set_column_widths(ws, widths)
    ws.freeze_panes = "A2"
    if records:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{len(records) + 1}"


# ------------------------------------------------------------------
# "Kontroll solgte biler"
# ------------------------------------------------------------------


def skriv_kontroll_solgte_biler(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll solgte biler"[:31])
    rows = bundle["kontroll_rows"]
    set_title_bar(ws, 1, 12, "Kontroll solgte biler")
    set_explanation_row(
        ws, 2, 12,
        "Kontrollregel: Bokfort dato sjekkes mot den OFV-transaksjonen som "
        "ligger NAERMEST bokfort dato i tid (for eller etter). Dager avvik "
        "er antall dager mellom denne datoen og bokfort dato. Har OFV ingen "
        "transaksjoner for kjoretoyet, brukes forstegangsregistrering fra "
        "Statens vegvesen (SVV) i stedet. Er OrgNrSelger fylt ut i input, "
        "flagges Selvhandel - det tvinger Dager avvik rod uansett faktisk "
        "avvik. Sortert med storst avvik forst."
    )

    bucket0 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and isinstance(r["DagerAvvik"], (int, float))
                  and r["Selvhandel"] != "Ja" and r["DagerAvvik"] <= AVVIK_GRONN_MAX)
    bucket1 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and isinstance(r["DagerAvvik"], (int, float))
                  and r["Selvhandel"] != "Ja"
                  and AVVIK_GRONN_MAX < r["DagerAvvik"] <= AVVIK_GUL_MAX)
    bucket2 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and (r["Selvhandel"] == "Ja"
                       or not isinstance(r["DagerAvvik"], (int, float))
                       or r["DagerAvvik"] > AVVIK_GUL_MAX))

    set_kpi_box(ws, 4, 1, 2, "Inputbiler", len(rows), big=True)
    set_kpi_box(ws, 4, 3, 4, "0-2 dager", bucket0, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, 4, 5, 6, "3-14 dager", bucket1, FARGE_VARSEL_BG, FARGE_VARSEL_FONT, big=True)
    set_kpi_box(ws, 4, 7, 8, "15+ dager / selvhandel", bucket2, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)

    header_row = 7
    headers = ["Kilde", "API-treff", "Regnr", "Chassisnummer", "Modell",
               "Forstegangsregistrert", "Bokfort dato", "Transaksjonsdato",
               "RegistreringsType", "Dager avvik", "Selger", "Kjoper"]
    style_table_header(ws, header_row, headers)

    r = header_row + 1
    for row in rows:
        ws.cell(row=r, column=1, value=row["Kilde"])
        ws.cell(row=r, column=2, value=row["ApiTreff"])
        ws.cell(row=r, column=3, value=row["RegnrInput"])
        ws.cell(row=r, column=4, value=row["Chassisnummer"])
        ws.cell(row=r, column=5, value=row["Modell"])
        if row["Forstegangsregistrert"]:
            ws.cell(row=r, column=6, value=row["Forstegangsregistrert"]).number_format = "dd.mm.yyyy"
        if row["BokfortDato"]:
            ws.cell(row=r, column=7, value=row["BokfortDato"]).number_format = "dd.mm.yyyy"
        if row["KontrollTransaksjonDato"]:
            ws.cell(row=r, column=8, value=row["KontrollTransaksjonDato"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=9, value=row["RegistreringsType"])

        dager = row["DagerAvvik"]
        if isinstance(dager, (int, float)):
            avvik_cell = ws.cell(row=r, column=10, value=dager)
            if row["Selvhandel"] == "Ja" or dager > AVVIK_GUL_MAX:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_AVVIK_BG)
                avvik_cell.font = Font(color=FARGE_AVVIK_FONT, bold=True)
            elif dager <= AVVIK_GRONN_MAX:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_OK_BG)
                avvik_cell.font = Font(color=FARGE_OK_FONT, bold=True)
            else:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_VARSEL_BG)
                avvik_cell.font = Font(color=FARGE_VARSEL_FONT, bold=True)

        ws.cell(row=r, column=11, value=row["Selger"])
        ws.cell(row=r, column=12, value=row["Kjoper"])
        r += 1

    set_column_widths(ws, {
        "A": 10, "B": 24, "C": 14, "D": 22, "E": 18, "F": 16, "G": 16,
        "H": 16, "I": 24, "J": 12, "K": 25, "L": 25,
    })
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)
    if rows:
        ws.auto_filter.ref = f"A{header_row}:L{header_row + len(rows)}"


# ------------------------------------------------------------------
# "Kontroll Varekjop Bruktbil"
# ------------------------------------------------------------------


def _skriv_bil_rad(ws, r: int, row: dict) -> None:
    ws.cell(row=r, column=1, value=row["RegnrInput"])
    ws.cell(row=r, column=2, value=row["Chassisnummer"])
    ws.cell(row=r, column=3, value=row["Modell"])
    ws.cell(row=r, column=4, value=row["FlaggVerdi"])
    if row["FraDato"]:
        ws.cell(row=r, column=5, value=row["FraDato"]).number_format = "dd.mm.yyyy"
    if row["TilDato"]:
        ws.cell(row=r, column=6, value=row["TilDato"]).number_format = "dd.mm.yyyy"
    ws.cell(row=r, column=7, value=row["KjoptFra"])
    if row["TilDato"]:
        ws.cell(row=r, column=8, value=row["Kjoper"])
    else:
        ws.cell(row=r, column=8, value="(fortsatt pa lager)")
    status_cell = ws.cell(row=r, column=9, value=row["Status"])
    color_status_cell(status_cell, row["Status"])


def skriv_kontroll_varekjop(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll Varekjop Bruktbil"[:31])
    seksjon_a, seksjon_b, seksjon_c = bundle["seksjon_a"], bundle["seksjon_b"], bundle["seksjon_c"]
    buyer_org_no = bundle["orgnr"]

    buyer_org_name = finn_orgnavn(bundle["all_rows"], buyer_org_no)

    set_title_bar(ws, 1, 9, "Kontroll Varekjop Bruktbil")
    set_explanation_row(
        ws, 2, 9,
        f"Kontrollregel: Biler i Innkjop-listen ({bundle['dato_fra']:%d.%m.%Y} - "
        f"{bundle['dato_til']:%d.%m.%Y}) kontrolleres mot OFV sin "
        f"transaksjonshistorikk for {buyer_org_name} (orgnr {buyer_org_no}): var "
        "bilen pa lager hos forhandleren pa et tidspunkt i perioden (kjopt av "
        "forhandleren, og eventuelt ikke solgt videre for perioden startet), og "
        "er alt OFV sier er kjopt i perioden faktisk bokfort."
    )
    set_explanation_row(
        ws, 3, 9,
        "Seksjon B og C er tillegg og vises bare hvis IB- og/eller UB-listen "
        "er fylt ut: biler kjopt forrige periode (IB) kontrolleres mot om de "
        "er solgt i denne perioden, og biler fortsatt pa lager (UB) "
        "kontrolleres mot om de kan forklares av IB eller innkjop i perioden."
    )

    r = 5
    bekreftet = sum(1 for x in seksjon_a if x["Status"].startswith("OK"))
    avvik = sum(1 for x in seksjon_a if x["Status"].startswith("Avvik") and "mangler i bokforing" not in x["Status"])
    mangler = sum(1 for x in seksjon_a if "mangler i bokforing" in x["Status"])

    set_section_bar(ws, r, 9, "A - Innkjop i perioden")
    r += 2
    set_kpi_box(ws, r - 1, 1, 2, "Innkjop", len(seksjon_a), big=True)
    set_kpi_box(ws, r - 1, 3, 4, "OK - pa lager i perioden", bekreftet, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, r - 1, 5, 6, "Avvik", avvik, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
    set_kpi_box(ws, r - 1, 7, 8, "OFV-kjop uten bokforing", mangler, FARGE_VARSEL_BG, FARGE_VARSEL_FONT, big=True)
    r += 1

    set_explanation_row(ws, r, 9, f"Forhandler: {buyer_org_name}")
    r += 1
    style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "I OFV-liste",
                               "Pa lager fra", "Pa lager til", "Kjopt fra", "Solgt til", "Status"])
    r += 1
    for row in seksjon_a:
        _skriv_bil_rad(ws, r, row)
        r += 1

    r += 1
    if seksjon_b:
        ok_ib = sum(1 for x in seksjon_b if x["Status"].startswith("OK"))
        solgt_for = sum(1 for x in seksjon_b if x["Status"] == "Avvik: solgt for perioden startet")
        ikke_funnet = sum(1 for x in seksjon_b if x["Status"] == "Avvik: fant ikke opprinnelig kjop hos forhandleren")

        set_section_bar(ws, r, 9, "B - IB (kjopt forrige periode)")
        r += 2
        set_kpi_box(ws, r - 1, 1, 2, "IB-biler", len(seksjon_b), big=True)
        set_kpi_box(ws, r - 1, 3, 4, "OK - pa lager i perioden", ok_ib, FARGE_OK_BG, FARGE_OK_FONT, big=True)
        set_kpi_box(ws, r - 1, 5, 6, "Avvik: solgt for perioden", solgt_for, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        set_kpi_box(ws, r - 1, 7, 8, "Avvik: fant ikke kjop", ikke_funnet, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        r += 1

        set_explanation_row(ws, r, 9, f"Forhandler: {buyer_org_name}")
        r += 1
        style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "Pa UB-liste",
                                   "Pa lager fra", "Pa lager til", "Kjopt fra", "Solgt til", "Status"])
        r += 1
        for row in seksjon_b:
            _skriv_bil_rad(ws, r, row)
            r += 1
        r += 1

    if seksjon_c:
        forklart = sum(1 for x in seksjon_c if x["Status"] == "OK - forklart")
        uforklart = len(seksjon_c) - forklart

        set_section_bar(ws, r, 9, "C - UB (fortsatt pa lager)")
        r += 2
        set_kpi_box(ws, r - 1, 1, 3, "UB-biler", len(seksjon_c), big=True)
        set_kpi_box(ws, r - 1, 4, 5, "Forklart", forklart, FARGE_OK_BG, FARGE_OK_FONT, big=True)
        set_kpi_box(ws, r - 1, 6, 8, "Uforklart", uforklart, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        r += 1
        style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "Funnet i IB", "Funnet i innkjop", "Status"])
        r += 1
        for row in seksjon_c:
            ws.cell(row=r, column=1, value=row["RegnrInput"])
            ws.cell(row=r, column=2, value=row["Chassisnummer"])
            ws.cell(row=r, column=3, value=row["Modell"])
            ws.cell(row=r, column=4, value=row["FunnetIIB"])
            ws.cell(row=r, column=5, value=row["FunnetIInnkjop"])
            status_cell = ws.cell(row=r, column=6, value=row["Status"])
            color_status_cell(status_cell, row["Status"])
            r += 1

    set_column_widths(ws, {
        "A": 14, "B": 22, "C": 18, "D": 16, "E": 18, "F": 18, "G": 25, "H": 25, "I": 34,
    })


# ------------------------------------------------------------------
# "Kontroll Demobil"
# ------------------------------------------------------------------


def skriv_kontroll_demobil(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll Demobil"[:31])
    rows = bundle["kontroll_rows"]
    set_title_bar(ws, 1, 7, "Kontroll Demobil")
    set_explanation_row(
        ws, 2, 7,
        "Kontrollregel: For hver bil sjekkes bilens NYESTE registrerte "
        f"OFV-transaksjon (uansett dato) - kjoperen der skal vaere juridisk "
        f"enhet (orgnr {bundle['orgnr']}). Er kjoperen et annet selskap "
        "eller en privatperson, er bilen sannsynligvis solgt videre og "
        "flagges som avvik."
    )

    ok = sum(1 for r in rows if r["Status"].startswith("OK"))
    avvik = sum(1 for r in rows if r["Status"].startswith("Avvik"))
    set_kpi_box(ws, 4, 1, 2, "Inputbiler", len(rows), big=True)
    set_kpi_box(ws, 4, 3, 4, "Fortsatt hos enhet", ok, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, 4, 5, 6, "Avvik (videresolgt)", avvik, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)

    header_row = 7
    style_table_header(ws, header_row, ["Regnr", "Chassisnummer", "Modell",
                                        "Bokfort inn dato", "Siste transaksjonsdato",
                                        "Siste kjoper", "Status"])

    r = header_row + 1
    for row in rows:
        ws.cell(row=r, column=1, value=row["RegnrInput"])
        ws.cell(row=r, column=2, value=row["Chassisnummer"])
        ws.cell(row=r, column=3, value=row["Modell"])
        if row["BokfortInnDato"]:
            ws.cell(row=r, column=4, value=row["BokfortInnDato"]).number_format = "dd.mm.yyyy"
        if row["SisteTransaksjonsDato"]:
            ws.cell(row=r, column=5, value=row["SisteTransaksjonsDato"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=6, value=row["SisteKjoper"])
        status_cell = ws.cell(row=r, column=7, value=row["Status"])
        color_status_cell(status_cell, row["Status"])
        r += 1

    set_column_widths(ws, {"A": 14, "B": 22, "C": 18, "D": 18, "E": 18, "F": 25, "G": 38})
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)
    if rows:
        ws.auto_filter.ref = f"A{header_row}:G{header_row + len(rows)}"


# ------------------------------------------------------------------
# Bygg hele arbeidsboken
# ------------------------------------------------------------------

_wb = Workbook()
_wb.remove(_wb.active)
_bundles = [b for b in (resultat_solgte_biler, resultat_varekjop, resultat_demobil) if b]

skriv_input_fane(_wb, _bundles)
if resultat_solgte_biler:
    skriv_resultat_fane(_wb, "Resultat solgte biler", resultat_solgte_biler["all_rows"])
    skriv_kontroll_solgte_biler(_wb, resultat_solgte_biler)
if resultat_varekjop:
    skriv_resultat_fane(_wb, "Resultat Varekjop", resultat_varekjop["all_rows"])
    skriv_kontroll_varekjop(_wb, resultat_varekjop)
if resultat_demobil:
    skriv_resultat_fane(_wb, "Resultat Demobil", resultat_demobil["all_rows"])
    skriv_kontroll_demobil(_wb, resultat_demobil)

_wb.save(OUTPUT_XLSX)
print(f"Excel-arbeidsbok lagret: {OUTPUT_XLSX}")

In [ ]:
# ==================================================================
# CELLE 5: HTML-RAPPORT
# ==================================================================
#
# En selvstendig HTML-fil (ingen eksterne avhengigheter - virker uten
# nett) med en fane per kontroll + Oversikt og Analyse. Alle tabeller kan
# sorteres, sokes i og filtreres pa hver kolonne, og filtrert utvalg kan
# eksporteres til CSV. Dataene legges inn som JSON og tegnes av
# JavaScript i nettleseren - Python bygger bare datamodellen under.
# Selve HTML/CSS/JS-malen ligger i _HTML_MAL nederst i cellen.

# ------------------------------------------------------------------
# Hjelpere for datamodellen
# ------------------------------------------------------------------


def _json_verdi(v):
    """Gjor en celleverdi JSON-vennlig (datoer som ISO, bool som Ja/Nei)."""
    if _er_tom(v):
        return None
    if isinstance(v, bool):
        return "Ja" if v else "Nei"
    if isinstance(v, datetime):
        return v.date().isoformat()
    if isinstance(v, date):
        return v.isoformat()
    if isinstance(v, (int, float)):
        return v
    return str(v)


def _kol(key: str, label: str, typ: str = "text", **ekstra) -> dict:
    """Kolonnedefinisjon. typ: text | num | date | status."""
    return {"key": key, "label": label, "type": typ, **ekstra}


def _tabell(tid: str, tittel: str, kolonner: list, rader: list,
            beskrivelse: str = "", ekstra_felt: tuple = ()) -> dict:
    keys = [k["key"] for k in kolonner] + list(ekstra_felt)
    return {
        "id": tid,
        "tittel": tittel,
        "beskrivelse": beskrivelse,
        "kolonner": kolonner,
        "rader": [{k: _json_verdi(r.get(k)) for k in keys} for r in rader],
    }


# Visningsnavn i HTML (bundle["kontroll"] er intern nokkel og endres ikke)
_VISNINGSNAVN = {
    "Kontroll solgte biler": "Solgte biler",
    "Varekjop bruktbil": "Varekjøp bruktbil",
    "Kontroll Demobil": "Demobil",
}


def _dager_mellom(fra, til):
    if isinstance(fra, date) and isinstance(til, date):
        return (til - fra).days
    return None


def _fmt_dato(d) -> str:
    return d.strftime("%d.%m.%Y") if isinstance(d, date) else ""


def _periode_tekst(bundle: dict) -> str:
    if bundle.get("dato_fra") and bundle.get("dato_til"):
        return f"{_fmt_dato(bundle['dato_fra'])} - {_fmt_dato(bundle['dato_til'])}"
    return ""


# Full OFV-historikk (tilsvarer Resultat-fanene i Excel). I HTML vises
# bil-nivaa-feltene pa ALLE rader, slik at filtrering pa f.eks. Merke
# ikke mister rader.
_HISTORIKK_KOLONNER = [
    _kol("Input", "Input"),
    _kol("Kilde", "Oppslag"),
    _kol("RegNo", "Regnr"),
    _kol("ChassisNumber", "Chassisnummer"),
    _kol("MakeName", "Merke"),
    _kol("ModelName", "Modell"),
    _kol("RegistrationType", "Registreringstype"),
    _kol("FuelGroup", "Drivstoff"),
    _kol("IsLeased", "Leaset"),
    _kol("IsUsedImported", "Bruktimport"),
    _kol("FirstRegistrationDate", "Førstegangsreg.", "date"),
    _kol("TransactionNumber", "Transaksjonsnr"),
    _kol("TransactionDate", "Eierskiftedato", "date"),
    _kol("SelgerType", "Selger"),
    _kol("FromOwnerType", "Selger eiertype"),
    _kol("FromOwnerOrgNo", "Selger orgnr"),
    _kol("FromOwnerCounty", "Selger fylke"),
    _kol("KjoperType", "Kjøper"),
    _kol("ToOwnerType", "Kjøper eiertype"),
    _kol("ToOwnerOrgNo", "Kjøper orgnr"),
    _kol("ToOwnerCounty", "Kjøper fylke"),
    _kol("Status", "Status", "status"),
]


def _historikk_tabell(tid: str, bundle: dict) -> dict:
    return _tabell(
        tid, "Full OFV-transaksjonshistorikk", _HISTORIKK_KOLONNER, bundle["all_rows"],
        "Alle eierskifter OFV returnerte for bilene i kontrollen (samme innhold "
        "som Resultat-fanen i Excel), nyeste først per bil.",
    )


def _vurdering_solgt(r: dict) -> tuple:
    """(tekst, tone) - samme inndeling som KPI-boksene i Excel."""
    if r["Kontrollert"] != "Ja":
        return "Ikke kontrollert", "neutral"
    if r["Selvhandel"] == "Ja":
        return "Avvik - selvhandel", "bad"
    dager = r["DagerAvvik"]
    if not isinstance(dager, (int, float)):
        return "Avvik", "bad"
    if dager <= AVVIK_GRONN_MAX:
        return f"OK (0-{AVVIK_GRONN_MAX} dager)", "ok"
    if dager <= AVVIK_GUL_MAX:
        return f"Varsel ({AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager)", "warn"
    return f"Avvik ({AVVIK_GUL_MAX + 1}+ dager)", "bad"


def _status_tone(status: str) -> str:
    status = status or ""
    if status.startswith("OK"):
        return "ok"
    if status.startswith("Avvik"):
        return "bad"
    if status.startswith("Varsel"):
        return "warn"
    return "neutral"


# ------------------------------------------------------------------
# En fane per kontroll
# ------------------------------------------------------------------


def _fane_solgte_biler(b: dict) -> dict:
    rader = []
    for r in b["kontroll_rows"]:
        tekst, tone = _vurdering_solgt(r)
        rader.append({**r, "Vurdering": tekst, "_tone": tone})

    teller = Counter(r["_tone"] for r in rader)
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])

    kolonner = [
        _kol("Vurdering", "Vurdering", "status"),
        _kol("DagerAvvik", "Dager avvik", "num", toneKey="_tone"),
        _kol("RegnrInput", "Regnr"),
        _kol("Chassisnummer", "Chassisnummer"),
        _kol("Modell", "Modell"),
        _kol("BokfortDato", "Bokført dato", "date"),
        _kol("KontrollTransaksjonDato", "Transaksjonsdato", "date"),
        _kol("RegistreringsType", "Registreringstype"),
        _kol("Forstegangsregistrert", "Førstegangsreg.", "date"),
        _kol("Selger", "Selger"),
        _kol("Kjoper", "Kjøper"),
        _kol("Kilde", "Kilde"),
        _kol("ApiTreff", "API-treff"),
        _kol("Kontrollert", "Kontrollert"),
    ]
    if b["orgnr"]:
        kolonner.insert(12, _kol("Selvhandel", "Selvhandel"))

    return {
        "id": "solgte",
        "tittel": "Solgte biler",
        "fullTittel": "Kontroll solgte biler",
        "regel": (
            "Bokført dato sjekkes mot den OFV-transaksjonen som ligger nærmest "
            "bokført dato i tid (før eller etter). Dager avvik er antall dager mellom "
            "de to datoene. Har OFV ingen transaksjoner for bilen, brukes "
            "førstegangsregistrering fra Statens vegvesen (SVV). Er orgnr for selger "
            "oppgitt, flagges selvhandel (samme selskap som selger og kjøper) som avvik "
            f"uansett dager. Grønn: 0-{AVVIK_GRONN_MAX} dager, gul: "
            f"{AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager, rød: {AVVIK_GUL_MAX + 1}+ dager."
        ),
        "meta": [
            ("Orgnr selger", b["orgnr"] or "(ikke oppgitt - ingen selvhandel-sjekk)"),
            ("Selskap", orgnavn),
            ("Kildefil", b["kildefil"]),
        ],
        "kpier": [
            {"label": "Biler i input", "verdi": len(rader), "tone": ""},
            {"label": f"OK 0-{AVVIK_GRONN_MAX} dager", "verdi": teller["ok"], "tone": "ok"},
            {"label": f"Varsel {AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager", "verdi": teller["warn"], "tone": "warn"},
            {"label": f"Avvik {AVVIK_GUL_MAX + 1}+ dager / selvhandel", "verdi": teller["bad"], "tone": "bad"},
            {"label": "Ikke kontrollert", "verdi": teller["neutral"], "tone": "neutral"},
            {"label": "Treff via SVV", "verdi": sum(1 for r in rader if r["Kilde"] == "SVV"), "tone": ""},
        ],
        "tabeller": [
            _tabell("solgte-kontroll", "Kontrollresultat", kolonner, rader,
                    "Én rad per bil i input, sortert med størst avvik først.",
                    ekstra_felt=("_tone",)),
            _historikk_tabell("solgte-historikk", b),
        ],
    }


_VAREKJOP_KOLONNER = [
    _kol("Status", "Status", "status"),
    _kol("RegnrInput", "Regnr"),
    _kol("Chassisnummer", "Chassisnummer"),
    _kol("Modell", "Modell"),
    _kol("FlaggVerdi", "FLAGG"),
    _kol("FraDato", "På lager fra", "date"),
    _kol("TilDato", "På lager til", "date"),
    _kol("Lagerdager", "Dager på lager", "num"),
    _kol("KjoptFra", "Kjøpt fra"),
    _kol("SolgtTil", "Solgt til"),
]


def _varekjop_rader(rader: list, dato_til: date) -> list:
    ut = []
    for r in rader:
        ut.append({
            **r,
            # Solgte biler: kjop -> salg. Fortsatt pa lager: kjop -> periodeslutt.
            "Lagerdager": _dager_mellom(r["FraDato"], r["TilDato"] or dato_til),
            "SolgtTil": r["Kjoper"] if r["TilDato"] else "(fortsatt pa lager)",
        })
    return ut


def _fane_varekjop(b: dict) -> dict:
    a, bb, c = b["seksjon_a"], b["seksjon_b"], b["seksjon_c"]
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])

    def kolonner_med_flagg(etikett):
        return [dict(k, label=etikett) if k["key"] == "FlaggVerdi" else k
                for k in _VAREKJOP_KOLONNER]

    tabeller = [_tabell(
        "varekjop-a", "A - Innkjøp i perioden", kolonner_med_flagg("I OFV-kjøpsliste"),
        _varekjop_rader(a, b["dato_til"]),
        "Biler i Innkjøp-listen: var bilen på lager hos forhandleren i perioden? "
        "Inkluderer også biler OFV sier er kjøpt i perioden, men som mangler i "
        "bokføringen. Dager på lager regnes til salg, eller til periodeslutt hvis "
        "bilen fortsatt er på lager.",
    )]
    if bb:
        tabeller.append(_tabell(
            "varekjop-b", "B - IB (kjøpt forrige periode)", kolonner_med_flagg("På UB-liste"),
            _varekjop_rader(bb, b["dato_til"]),
            "Biler kjøpt før perioden startet: OK hvis bilen fortsatt var på lager "
            "når perioden startet.",
        ))
    if c:
        tabeller.append(_tabell(
            "varekjop-c", "C - UB (fortsatt på lager)",
            [_kol("Status", "Status", "status"), _kol("RegnrInput", "Regnr"),
             _kol("Chassisnummer", "Chassisnummer"), _kol("Modell", "Modell"),
             _kol("FunnetIIB", "Funnet i IB"), _kol("FunnetIInnkjop", "Funnet i innkjøp")],
            c, "Biler fortsatt på lager - forklart hvis de finnes i IB eller innkjøp.",
        ))
    tabeller.append(_historikk_tabell("varekjop-historikk", b))

    s = b["stats"]
    kpier = [
        {"label": "Innkjøp i input", "verdi": s["Innkjop (antall)"], "tone": ""},
        {"label": "OK - på lager i perioden", "verdi": s["Innkjop - OK"], "tone": "ok"},
        {"label": "Avvik", "verdi": s["Innkjop - Avvik"], "tone": "bad"},
        {"label": "OFV-kjøp uten bokføring", "verdi": s["OFV-kjop uten bokforing"], "tone": "warn"},
    ]
    if bb:
        kpier.append({"label": "IB-biler", "verdi": len(bb), "tone": ""})
    if c:
        kpier.append({"label": "UB uforklart", "verdi": sum(
            1 for r in c if r["Status"].startswith("Avvik")), "tone": "bad"})

    return {
        "id": "varekjop",
        "tittel": "Varekjøp bruktbil",
        "fullTittel": "Kontroll Varekjøp bruktbil",
        "regel": (
            "For hver bil beregnes lagerperioden hos forhandleren: siste kjøp av "
            "oppgitt juridisk enhet (På lager fra) og første videresalg etter det "
            "kjøpet (På lager til). OK så lenge lagerperioden overlapper "
            "kontrollperioden. I tillegg hentes alle biler OFV sier er kjøpt av "
            "enheten i perioden - de som ikke står i Innkjøp-listen flagges som "
            "«mangler i bokføring»."
        ),
        "meta": [
            ("Orgnr kjøper", b["orgnr"]),
            ("Selskap", orgnavn),
            ("Periode", _periode_tekst(b)),
            ("Kildefil", b["kildefil"]),
        ],
        "kpier": kpier,
        "tabeller": tabeller,
    }


def _fane_demobil(b: dict) -> dict:
    rader = b["kontroll_rows"]
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])
    i_dag = date.today()
    rader = [{**r, "DagerSidenBokfort": _dager_mellom(r["BokfortInnDato"], i_dag),
              "DagerSidenSisteTx": _dager_mellom(r["SisteTransaksjonsDato"], i_dag)}
             for r in rader]
    s = b["stats"]
    meta = [("Orgnr", b["orgnr"]), ("Selskap", orgnavn)]
    if _periode_tekst(b):
        meta.append(("Periode", _periode_tekst(b)))
    meta.append(("Kildefil", b["kildefil"]))

    return {
        "id": "demobil",
        "tittel": "Demobil",
        "fullTittel": "Kontroll Demobil",
        "regel": (
            "En demobil skal være registrert på selskapets juridiske enhet. For hver "
            "bil sjekkes bilens NYESTE registrerte OFV-transaksjon (uansett dato): er "
            "kjøperen der oppgitt orgnr, er bilen fortsatt eid av enheten (OK). Er "
            "kjøperen et annet selskap eller en privatperson, er bilen sannsynligvis "
            "solgt videre (Avvik)."
        ),
        "meta": meta,
        "kpier": [
            {"label": "Biler i input", "verdi": s["Antall biler"], "tone": ""},
            {"label": "Fortsatt hos enhet", "verdi": s["Fortsatt hos enhet"], "tone": "ok"},
            {"label": "Avvik (videresolgt)", "verdi": s["Avvik (videresolgt)"], "tone": "bad"},
            {"label": "Ingen OFV-treff", "verdi": s["Ingen OFV-treff"], "tone": "neutral"},
        ],
        "tabeller": [
            _tabell("demobil-kontroll", "Kontrollresultat", [
                _kol("Status", "Status", "status"),
                _kol("EidAvEnhet", "Eid av enhet"),
                _kol("RegnrInput", "Regnr"),
                _kol("Chassisnummer", "Chassisnummer"),
                _kol("Modell", "Modell"),
                _kol("BokfortInnDato", "Bokført inn", "date"),
                _kol("DagerSidenBokfort", "Dager siden bokført inn", "num"),
                _kol("SisteTransaksjonsDato", "Siste eierskifte", "date"),
                _kol("DagerSidenSisteTx", "Dager siden siste eierskifte", "num"),
                _kol("SisteKjoper", "Siste kjøper"),
            ], rader, "Én rad per demobil i input."),
            _historikk_tabell("demobil-historikk", b),
        ],
    }


# ------------------------------------------------------------------
# Oversikt og Analyse
# ------------------------------------------------------------------


def _input_tabell(bundles: list) -> dict:
    rader = []
    for b in bundles:
        for bil in b["biler_manifest"]:
            rader.append({
                "Kontroll": _VISNINGSNAVN.get(b["kontroll"], b["kontroll"]),
                "Identifikator": bil["identifikator"],
                "Type": "VIN" if bil["vin"] else "Regnr",
                "BokfortDato": parse_bokfort_dato(bil.get("BokfortDato")),
                "Liste": bil.get("Liste"),
                "BokfortInnDato": parse_bokfort_dato(bil.get("BokfortInnDato")),
            })
    return _tabell("input-biler", "Biler lest fra inputfilene", [
        _kol("Kontroll", "Kontroll"),
        _kol("Identifikator", "Regnr/VIN"),
        _kol("Type", "Type"),
        _kol("BokfortDato", "Bokført dato", "date"),
        _kol("Liste", "Liste"),
        _kol("BokfortInnDato", "Bokført inn dato", "date"),
    ], rader, "Akkurat det notebooken leste fra Input-mappen, én rad per bil per kontroll.")


def _topp(teller: Counter, n: int = 10, annet: str = "Andre") -> list:
    """Topp-n kategorier, resten slatt sammen til en 'Andre'-rad."""
    data = [{"label": k, "value": v} for k, v in teller.most_common(n)]
    rest = sum(teller.values()) - sum(d["value"] for d in data)
    if rest > 0:
        data.append({"label": annet, "value": rest, "muted": True})
    return data


def _bokser(verdier: list, grenser: list) -> list:
    """Teller verdier i intervaller. grenser = [(etikett, min, maks), ...]."""
    return [{"label": etikett, "value": sum(1 for v in verdier if lo <= v <= hi)}
            for etikett, lo, hi in grenser]


def _unike_kjoretoy(bundles: list) -> dict:
    """Ett representativt OFV-rad per kjoretoy (pa tvers av kontrollene)."""
    biler = {}
    for b in bundles:
        for r in b["all_rows"]:
            if r.get("Status") != "OK":
                continue
            key = normalize_identifier(r.get("ChassisNumber")) or normalize_identifier(r.get("RegNo"))
            if key and key not in biler:
                biler[key] = r
    return biler


def _unike_transaksjoner(bundles: list) -> list:
    sett, ut = set(), []
    for b in bundles:
        for r in b["all_rows"]:
            if r.get("Status") != "OK" or not r.get("TransactionDate"):
                continue
            key = (r.get("TransactionNumber"), r.get("ChassisNumber"), r.get("TransactionDate"))
            if key not in sett:
                sett.add(key)
                ut.append(r)
    return ut


def _analyse(bundles: list, faner: list) -> dict:
    grafer = []

    # 1. Status per kontroll (100 % stablet)
    stabel = []
    for b in bundles:
        if b["kontroll"] == "Kontroll solgte biler":
            t = Counter(_vurdering_solgt(r)[1] for r in b["kontroll_rows"])
            seg = [("OK", t["ok"], "ok"), ("Varsel", t["warn"], "warn"),
                   ("Avvik", t["bad"], "bad"), ("Ikke kontrollert", t["neutral"], "neutral")]
        elif b["kontroll"] == "Varekjop bruktbil":
            rows = b["seksjon_a"] + b["seksjon_b"] + b["seksjon_c"]
            mangler = sum(1 for r in rows if "mangler i bokforing" in r["Status"])
            seg = [("OK", sum(1 for r in rows if r["Status"].startswith("OK")), "ok"),
                   ("Mangler bokføring", mangler, "warn"),
                   ("Avvik", sum(1 for r in rows if r["Status"].startswith("Avvik")) - mangler, "bad")]
        else:
            t = Counter(_status_tone(r["Status"]) for r in b["kontroll_rows"])
            seg = [("OK", t["ok"], "ok"), ("Avvik", t["bad"], "bad"),
                   ("Ingen treff", t["neutral"], "neutral")]
        stabel.append({"label": _VISNINGSNAVN.get(b["kontroll"], b["kontroll"]),
                       "segments": [{"label": l, "value": v, "tone": tone} for l, v, tone in seg]})
    grafer.append({"type": "stack", "tittel": "Status per kontroll",
                   "undertittel": "Andel biler per utfall - hold musen over for antall",
                   "rows": stabel, "bred": True})

    for b in bundles:
        if b["kontroll"] == "Kontroll solgte biler":
            dager = [r["DagerAvvik"] for r in b["kontroll_rows"]
                     if r["Kontrollert"] == "Ja" and isinstance(r["DagerAvvik"], (int, float))]
            if dager:
                data = _bokser(dager, [
                    ("0-2", 0, 2), ("3-7", 3, 7), ("8-14", 8, 14), ("15-30", 15, 30),
                    ("31-90", 31, 90), ("91-365", 91, 365), ("> 1 år", 366, 10 ** 9)])
                for d in data:
                    d["tone"] = "ok" if d["label"] == "0-2" else (
                        "warn" if d["label"] in ("3-7", "8-14") else "bad")
                grafer.append({"type": "columns", "tittel": "Solgte biler: dager avvik",
                               "undertittel": "Antall kontrollerte biler per avviksintervall (dager)",
                               "data": data, "statusLegend": True})
            grafer.append({"type": "hbar", "tittel": "Solgte biler: kilde for kontrolldato",
                           "undertittel": "OFV-eierskifte, SVV-førstegangsregistrering eller ingen treff",
                           "data": _topp(Counter(r["Kilde"] for r in b["kontroll_rows"]))})

        if b["kontroll"] == "Varekjop bruktbil":
            rows = b["seksjon_a"] + b["seksjon_b"]
            lagertid = [_dager_mellom(r["FraDato"], r["TilDato"]) for r in rows
                        if r["FraDato"] and r["TilDato"]]
            if lagertid:
                grafer.append({"type": "columns", "tittel": "Varekjøp: lagertid for solgte biler",
                               "undertittel": "Dager fra forhandlerens kjøp til videresalg",
                               "data": _bokser(lagertid, [
                                   ("0-30", 0, 30), ("31-60", 31, 60), ("61-90", 61, 90),
                                   ("91-180", 91, 180), ("181-365", 181, 365),
                                   ("> 1 år", 366, 10 ** 9)])})
            kjopt_fra = Counter(r["KjoptFra"] for r in rows if r["KjoptFra"])
            if kjopt_fra:
                grafer.append({"type": "hbar", "tittel": "Varekjøp: hvor kommer bilene fra",
                               "undertittel": "Selger i forhandlerens kjøp (topp 10)",
                               "data": _topp(kjopt_fra)})
            solgt_til = Counter(r["Kjoper"] for r in rows if r["TilDato"] and r["Kjoper"])
            if solgt_til:
                grafer.append({"type": "hbar", "tittel": "Varekjøp: hvem kjøper bilene videre",
                               "undertittel": "Kjøper ved videresalg (topp 10)",
                               "data": _topp(solgt_til)})

        if b["kontroll"] == "Kontroll Demobil":
            kjopere = Counter(r["SisteKjoper"] for r in b["kontroll_rows"]
                              if r["Status"].startswith("Avvik") and r["SisteKjoper"])
            if kjopere:
                grafer.append({"type": "hbar", "tittel": "Demobil: hvem har overtatt bilene",
                               "undertittel": "Siste kjøper for demobiler med avvik (topp 10)",
                               "data": _topp(kjopere)})
            alder = [_dager_mellom(parse_bokfort_dato(r["BokfortInnDato"]), date.today())
                     for r in b["kontroll_rows"] if r["BokfortInnDato"]]
            if alder:
                grafer.append({"type": "columns", "tittel": "Demobil: tid siden bokført inn",
                               "undertittel": "Antall demobiler etter alder i dager",
                               "data": _bokser(alder, [
                                   ("0-90", 0, 90), ("91-180", 91, 180), ("181-365", 181, 365),
                                   ("1-2 år", 366, 730), ("> 2 år", 731, 10 ** 9)])})

    # Pa tvers av kontrollene
    biler = _unike_kjoretoy(bundles)
    if biler:
        grafer.append({"type": "hbar", "tittel": "Merke",
                       "undertittel": f"Unike kjøretøy med OFV-treff (n = {len(biler)})",
                       "data": _topp(Counter((r.get("MakeName") or "Ukjent") for r in biler.values()))})
        grafer.append({"type": "hbar", "tittel": "Drivstoff",
                       "undertittel": "Unike kjøretøy med OFV-treff",
                       "data": _topp(Counter((r.get("FuelGroup") or "Ukjent") for r in biler.values()), 7)})

    tx = _unike_transaksjoner(bundles)
    if tx:
        siste = max(r["TransactionDate"] for r in tx)
        maneder = []
        y, m = siste.year, siste.month
        for _ in range(24):
            maneder.append((y, m))
            y, m = (y, m - 1) if m > 1 else (y - 1, 12)
        maneder.reverse()
        teller = Counter((r["TransactionDate"].year, r["TransactionDate"].month) for r in tx)
        grafer.append({"type": "columns", "tittel": "Eierskifter per måned",
                       "undertittel": "Unike OFV-transaksjoner for alle kontrollerte biler, siste 24 måneder",
                       "data": [{"label": f"{mm:02d}.{str(yy)[2:]}", "value": teller[(yy, mm)]}
                                for yy, mm in maneder], "bred": True})
        grafer.append({"type": "hbar", "tittel": "Kjøpertype i eierskifter",
                       "undertittel": "Eiertype på kjøpersiden, alle unike transaksjoner",
                       "data": _topp(Counter((r.get("ToOwnerType") or "Ukjent") for r in tx), 6)})

    tall = [
        {"label": "Biler kontrollert", "verdi": sum(b["antall_biler"] for b in bundles)},
        {"label": "Unike kjøretøy i OFV", "verdi": len(biler)},
        {"label": "Unike eierskifter", "verdi": len(tx)},
        {"label": "Avvik totalt", "verdi": sum(
            1 for f in faner for t in f["tabeller"] if not t["id"].endswith("historikk")
            for r in t["rader"]
            if str(r.get("Status") or r.get("Vurdering") or "").startswith("Avvik")), "tone": "bad"},
    ]
    return {"nokkeltall": tall, "grafer": grafer}


def bygg_rapportdata(bundles: list) -> dict:
    byggere = {
        "Kontroll solgte biler": _fane_solgte_biler,
        "Varekjop bruktbil": _fane_varekjop,
        "Kontroll Demobil": _fane_demobil,
    }
    faner = [byggere[b["kontroll"]](b) for b in bundles]
    for f in faner:
        f["meta"] = [{"label": k, "verdi": v} for k, v in f["meta"] if v]

    return {
        "selskap": SELSKAP,
        "generert": datetime.now().strftime("%d.%m.%Y kl. %H:%M"),
        "tidsbruk": round(_elapsed_s, 1),
        "api": {
            "ofvKjoretoy": API_TELLER["ofv_kjoretoy"],
            "ofvBuffer": API_TELLER["ofv_buffer"],
            "ofvHttp": API_TELLER["ofv_http"],
            "svvHttp": API_TELLER["svv_http"],
        },
        "oversikt": [{
            "id": f["id"], "tittel": f["fullTittel"], "meta": f["meta"],
            "kpier": f["kpier"],
        } for f in faner],
        "input": _input_tabell(bundles),
        "faner": faner,
        "analyse": _analyse(bundles, faner),
    }


def skriv_html_rapport(bundles: list, path: str) -> None:
    data = bygg_rapportdata(bundles)
    # "</" escapes sa JSON-innholdet aldri kan avslutte <script>-taggen.
    data_json = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
    tittel = html.escape(f"OFV-kontroller - {SELSKAP}")
    side = _HTML_MAL.replace("__TITTEL__", tittel).replace("__DATA__", data_json)
    with open(path, "w", encoding="utf-8") as f:
        f.write(side)


# ------------------------------------------------------------------
# HTML/CSS/JS-mal - __TITTEL__ og __DATA__ fylles inn av skriv_html_rapport
# ------------------------------------------------------------------

_HTML_MAL = r"""<!DOCTYPE html>
<html lang="no">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITTEL__</title>
<style>
:root {
  color-scheme: light;
  --bg: #f5f6f8;
  --surface: #ffffff;
  --surface-2: #f1f3f5;
  --surface-3: #e9ecef;
  --border: #e2e5e9;
  --border-strong: #cfd4da;
  --text: #24292f;
  --text-2: #5b6e7f;
  --muted: #8b97a3;
  --brand: #e81a3b;
  --series: #008fd2;
  --series-muted: #b9c3cc;
  --grid: #eceff2;
  --ok: #009966;   --ok-bg: #e3f4ed;   --ok-ink: #00704b;
  --warn: #d67900; --warn-bg: #fbefdf; --warn-ink: #9a5800;
  --bad: #e81a3b;  --bad-bg: #fde6ea;  --bad-ink: #b3122d;
  --neutral: #8b97a3; --neutral-bg: #eef0f2; --neutral-ink: #4d5b68;
  --shadow: 0 1px 2px rgba(16, 24, 32, .05), 0 1px 1px rgba(16, 24, 32, .03);
  --radius: 12px;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    color-scheme: dark;
    --bg: #111315; --surface: #1a1d20; --surface-2: #212529; --surface-3: #2a2f34;
    --border: #2c3136; --border-strong: #3a4046;
    --text: #e8eaed; --text-2: #aab6c1; --muted: #7d8994;
    --series: #2fa8e6; --series-muted: #4a535c; --grid: #262a2f;
    --ok: #22b37f; --ok-bg: #12352a; --ok-ink: #6fd8ad;
    --warn: #e8901f; --warn-bg: #3a2a12; --warn-ink: #f4b766;
    --bad: #ff4d63; --bad-bg: #3d1820; --bad-ink: #ff8b99;
    --neutral: #7d8994; --neutral-bg: #272c31; --neutral-ink: #b5c0ca;
    --shadow: none;
  }
}
:root[data-theme="dark"] {
  color-scheme: dark;
  --bg: #111315; --surface: #1a1d20; --surface-2: #212529; --surface-3: #2a2f34;
  --border: #2c3136; --border-strong: #3a4046;
  --text: #e8eaed; --text-2: #aab6c1; --muted: #7d8994;
  --series: #2fa8e6; --series-muted: #4a535c; --grid: #262a2f;
  --ok: #22b37f; --ok-bg: #12352a; --ok-ink: #6fd8ad;
  --warn: #e8901f; --warn-bg: #3a2a12; --warn-ink: #f4b766;
  --bad: #ff4d63; --bad-bg: #3d1820; --bad-ink: #ff8b99;
  --neutral: #7d8994; --neutral-bg: #272c31; --neutral-ink: #b5c0ca;
  --shadow: none;
}
* { box-sizing: border-box; }
html, body { margin: 0; }
body {
  background: var(--bg); color: var(--text);
  font: 14px/1.5 "Inter", "Segoe UI", system-ui, -apple-system, Roboto, "Helvetica Neue", Arial, sans-serif;
  -webkit-font-smoothing: antialiased;
}
button, input, select { font: inherit; color: inherit; }
.num, td.t-num, td.t-date { font-variant-numeric: tabular-nums; }

/* ---------- Topp ---------- */
.topbar {
  background: var(--surface); border-bottom: 1px solid var(--border);
  position: sticky; top: 0; z-index: 20;
}
.topbar-inner { max-width: 1440px; margin: 0 auto; padding: 14px 24px 0; }
.brandrow { display: flex; align-items: center; gap: 14px; flex-wrap: wrap; }
.brandmark { width: 10px; height: 28px; background: var(--brand); border-radius: 2px; }
.brandrow h1 { font-size: 18px; margin: 0; font-weight: 650; letter-spacing: -.01em; }
.brandrow .company { color: var(--text-2); font-size: 14px; }
.brandrow .spacer { flex: 1; }
.iconbtn {
  border: 1px solid var(--border); background: var(--surface); border-radius: 8px;
  padding: 5px 10px; cursor: pointer; color: var(--text-2); font-size: 13px;
}
.iconbtn:hover { background: var(--surface-2); color: var(--text); }
.tabs { display: flex; gap: 4px; margin-top: 10px; overflow-x: auto; scrollbar-width: none; }
.tab {
  border: 0; background: none; padding: 10px 14px 12px; cursor: pointer; color: var(--text-2);
  border-bottom: 2px solid transparent; white-space: nowrap; font-weight: 550; display: flex; gap: 8px; align-items: center;
}
.tab:hover { color: var(--text); }
.tab[aria-selected="true"] { color: var(--text); border-bottom-color: var(--brand); }
.tab .badge {
  font-size: 11px; font-weight: 650; padding: 1px 7px; border-radius: 999px;
  background: var(--bad-bg); color: var(--bad-ink);
}
.tab .badge.zero { background: var(--ok-bg); color: var(--ok-ink); }

main { max-width: 1440px; margin: 0 auto; padding: 24px; }
section.panel { display: none; }
section.panel.active { display: block; }
.pagehead { margin-bottom: 18px; }
.pagehead h2 { font-size: 22px; margin: 0 0 6px; letter-spacing: -.015em; font-weight: 650; }
.pagehead p { margin: 0; color: var(--text-2); max-width: 980px; }

.meta { display: flex; flex-wrap: wrap; gap: 8px; margin: 12px 0 0; }
.meta span {
  background: var(--surface); border: 1px solid var(--border); border-radius: 999px;
  padding: 3px 11px; font-size: 12.5px; color: var(--text-2); max-width: 100%;
  overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
}
.meta b { color: var(--text); font-weight: 600; }

details.rule { margin-top: 12px; max-width: 980px; }
details.rule summary { cursor: pointer; color: var(--text-2); font-size: 13px; }
details.rule p { margin: 8px 0 0; color: var(--text-2); }

/* ---------- KPI ---------- */
.kpis { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: 12px; margin: 20px 0 24px; }
.kpi {
  background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius);
  padding: 14px 16px; box-shadow: var(--shadow); position: relative; text-align: left;
}
button.kpi { cursor: pointer; }
button.kpi:hover { border-color: var(--border-strong); }
.kpi .label { color: var(--text-2); font-size: 12.5px; display: flex; gap: 6px; align-items: center; }
.kpi .value { font-size: 28px; font-weight: 650; letter-spacing: -.02em; margin-top: 2px; }
.dot { width: 8px; height: 8px; border-radius: 50%; display: inline-block; flex: none; }
.dot.ok { background: var(--ok); } .dot.warn { background: var(--warn); }
.dot.bad { background: var(--bad); } .dot.neutral { background: var(--neutral); }
.kpi.active { outline: 2px solid var(--text); outline-offset: -1px; }

/* ---------- Kort ---------- */
.card {
  background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius);
  box-shadow: var(--shadow); margin-bottom: 20px; overflow: hidden;
}
.card-head { padding: 16px 18px 4px; }
.card-head h3 { margin: 0; font-size: 15px; font-weight: 650; }
.card-head p { margin: 2px 0 0; color: var(--text-2); font-size: 13px; }
details.card > summary { list-style: none; cursor: pointer; }
details.card > summary::-webkit-details-marker { display: none; }
details.card > summary .card-head { padding-bottom: 16px; display: flex; gap: 12px; align-items: center; }
details.card > summary .chev { transition: transform .15s; color: var(--muted); }
details.card[open] > summary .chev { transform: rotate(90deg); }
details.card[open] > summary .card-head { padding-bottom: 4px; }

.overview-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(380px, 1fr)); gap: 16px; margin: 20px 0 24px; }
.ov { padding: 18px; display: flex; flex-direction: column; gap: 12px; margin: 0; }
.ov h3 { margin: 0; font-size: 16px; }
.ov .minis { display: grid; grid-template-columns: repeat(auto-fill, minmax(100px, 1fr)); gap: 8px; }
.mini { background: var(--surface-2); border-radius: 8px; padding: 8px 10px; }
.mini .l { font-size: 11.5px; color: var(--text-2); display: flex; gap: 5px; align-items: center; line-height: 1.3; }
.mini .v { font-size: 20px; font-weight: 650; }
.ov .open { align-self: flex-start; border: 1px solid var(--border); background: var(--surface); border-radius: 8px; padding: 6px 12px; cursor: pointer; font-weight: 550; }
.ov .open:hover { background: var(--surface-2); }
.runinfo { display: flex; flex-wrap: wrap; gap: 8px 24px; color: var(--text-2); font-size: 13px; }
.runinfo b { color: var(--text); font-weight: 600; }

/* ---------- Tabell ---------- */
.toolbar { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; padding: 12px 18px; }
.search { position: relative; flex: 1 1 260px; max-width: 380px; }
.search input {
  width: 100%; padding: 7px 10px 7px 32px; border: 1px solid var(--border-strong); border-radius: 8px;
  background: var(--surface); outline: none;
}
.search input:focus, .filters input:focus, .filters select:focus { border-color: var(--series); box-shadow: 0 0 0 3px color-mix(in srgb, var(--series) 18%, transparent); }
.search svg { position: absolute; left: 10px; top: 50%; transform: translateY(-50%); color: var(--muted); }
.chips { display: flex; gap: 6px; flex-wrap: wrap; }
.chip {
  border: 1px solid var(--border); background: var(--surface); border-radius: 999px; padding: 4px 11px;
  cursor: pointer; font-size: 12.5px; display: inline-flex; gap: 6px; align-items: center; color: var(--text-2);
}
.chip:hover { background: var(--surface-2); }
.chip[aria-pressed="true"] { background: var(--text); color: var(--surface); border-color: var(--text); }
.chip .n { font-weight: 650; }
.toolbar .grow { flex: 1; }
.btn {
  border: 1px solid var(--border); background: var(--surface); border-radius: 8px; padding: 6px 11px;
  cursor: pointer; font-size: 12.5px; color: var(--text-2);
}
.btn:hover { background: var(--surface-2); color: var(--text); }
.colmenu { position: relative; }
.colmenu > summary { list-style: none; }
.colmenu > summary::-webkit-details-marker { display: none; }
.colmenu .menu {
  position: absolute; right: 0; top: calc(100% + 6px); z-index: 15; background: var(--surface);
  border: 1px solid var(--border); border-radius: 10px; padding: 8px; min-width: 220px; max-height: 340px; overflow: auto;
  box-shadow: 0 8px 24px rgba(0,0,0,.12);
}
.colmenu label { display: flex; gap: 8px; padding: 4px 6px; border-radius: 6px; cursor: pointer; font-size: 13px; }
.colmenu label:hover { background: var(--surface-2); }

.tablewrap { overflow: auto; max-height: 72vh; border-top: 1px solid var(--border); }
table { border-collapse: separate; border-spacing: 0; width: 100%; font-size: 13px; }
thead th {
  position: sticky; z-index: 2; background: var(--surface-2); text-align: left; font-weight: 600;
  color: var(--text-2); white-space: nowrap; border-bottom: 1px solid var(--border);
}
thead tr.labels th { top: 0; padding: 9px 12px 6px; cursor: pointer; user-select: none; font-size: 12px; }
thead tr.labels th:hover { color: var(--text); }
thead tr.labels th .arrow { opacity: .35; margin-left: 4px; font-size: 10px; }
thead tr.labels th.sorted .arrow { opacity: 1; color: var(--text); }
thead tr.filters th { top: 34px; padding: 0 8px 8px; }
.filters input, .filters select {
  width: 100%; min-width: 90px; border: 1px solid var(--border); border-radius: 6px; padding: 4px 6px;
  background: var(--surface); font-size: 12px; font-weight: 400; outline: none;
}
.filters .range { display: flex; gap: 4px; }
.filters .range input { min-width: 70px; }
.filters input[type="date"] { min-width: 118px; }
.filters .active { border-color: var(--series); background: color-mix(in srgb, var(--series) 7%, var(--surface)); }
tbody td { padding: 8px 12px; border-bottom: 1px solid var(--border); white-space: nowrap; max-width: 360px; overflow: hidden; text-overflow: ellipsis; }
tbody tr:hover td { background: var(--surface-2); }
td.t-num { text-align: right; }
td.t-status { max-width: none; }
td.empty { color: var(--muted); }
.pill {
  display: inline-flex; align-items: center; gap: 6px; padding: 2px 9px; border-radius: 999px;
  font-size: 12px; font-weight: 600; white-space: nowrap;
}
.pill.ok { background: var(--ok-bg); color: var(--ok-ink); }
.pill.warn { background: var(--warn-bg); color: var(--warn-ink); }
.pill.bad { background: var(--bad-bg); color: var(--bad-ink); }
.pill.neutral { background: var(--neutral-bg); color: var(--neutral-ink); }
.tablefoot { display: flex; gap: 12px; align-items: center; padding: 10px 18px; color: var(--text-2); font-size: 12.5px; border-top: 1px solid var(--border); }
.noresult { padding: 28px; text-align: center; color: var(--muted); }

/* ---------- Grafer ---------- */
.chartgrid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px; }
.chartgrid .card { margin: 0; }
.chartgrid .wide { grid-column: 1 / -1; }
@media (max-width: 980px) { .chartgrid { grid-template-columns: 1fr; } .overview-grid { grid-template-columns: 1fr; } }
.chartbody { padding: 8px 18px 6px; }
.chartbody svg { display: block; width: 100%; overflow: visible; }
.chartbody svg text { font-family: inherit; }
.legend { display: flex; flex-wrap: wrap; gap: 6px 16px; padding: 0 18px 4px; font-size: 12.5px; color: var(--text-2); }
.legend span { display: inline-flex; gap: 6px; align-items: center; }
.legend i { width: 10px; height: 10px; border-radius: 3px; display: inline-block; }
details.datatable { padding: 0 18px 14px; }
details.datatable summary { cursor: pointer; font-size: 12.5px; color: var(--text-2); }
details.datatable table { margin-top: 8px; font-size: 12.5px; }
details.datatable td, details.datatable th { padding: 4px 8px; border-bottom: 1px solid var(--border); text-align: left; position: static; background: none; }
details.datatable td.n { text-align: right; font-variant-numeric: tabular-nums; }
.tip {
  position: fixed; pointer-events: none; z-index: 50; background: var(--text); color: var(--surface);
  padding: 7px 10px; border-radius: 8px; font-size: 12.5px; line-height: 1.35; max-width: 280px;
  box-shadow: 0 6px 18px rgba(0,0,0,.18); opacity: 0; transition: opacity .08s;
}
.tip b { font-weight: 650; }
.tip.show { opacity: 1; }
footer { max-width: 1440px; margin: 0 auto; padding: 8px 24px 32px; color: var(--muted); font-size: 12px; }

@media (max-width: 640px) {
  .topbar-inner, main { padding-left: 16px; padding-right: 16px; }
  .kpi .value { font-size: 24px; }
}
@media print {
  .topbar { position: static; }
  .tabs, .toolbar, thead tr.filters, .tablefoot .btn, .iconbtn { display: none !important; }
  section.panel { display: block !important; break-before: page; }
  .tablewrap { max-height: none; overflow: visible; }
  details.card > *:not(summary) { display: block; }
  body { background: #fff; }
}
</style>
</head>
<body>
<header class="topbar">
  <div class="topbar-inner">
    <div class="brandrow">
      <span class="brandmark" aria-hidden="true"></span>
      <h1>OFV-kontroller</h1>
      <span class="company" id="hdr-company"></span>
      <span class="spacer"></span>
      <button class="iconbtn" id="btn-theme" type="button" title="Bytt mellom lyst og mørkt tema">Tema</button>
      <button class="iconbtn" id="btn-print" type="button" title="Skriv ut / lagre som PDF">Skriv ut</button>
    </div>
    <nav class="tabs" role="tablist" id="tabs"></nav>
  </div>
</header>
<main id="main"></main>
<footer id="footer"></footer>
<div class="tip" id="tip" role="tooltip"></div>
<script id="rapport-data" type="application/json">__DATA__</script>
<script>
(function () {
  "use strict";
  var D = JSON.parse(document.getElementById("rapport-data").textContent);
  var SIDE = 200;            // rader per "side" i tabellene
  var nf = new Intl.NumberFormat("nb-NO");
  var pf = new Intl.NumberFormat("nb-NO", { style: "percent", maximumFractionDigits: 0 });
  var coll = new Intl.Collator("nb", { numeric: true, sensitivity: "base" });
  var TONE_NAVN = { ok: "OK", warn: "Varsel", bad: "Avvik", neutral: "Annet" };

  // ---------- Sma hjelpere ----------
  function el(tag, attrs) {
    var e = document.createElement(tag);
    if (attrs) for (var k in attrs) {
      var v = attrs[k];
      if (v == null || v === false) continue;
      if (k === "text") e.textContent = v;
      else if (k === "cls") e.className = v;
      else if (k.slice(0, 2) === "on") e.addEventListener(k.slice(2), v);
      else e.setAttribute(k, v === true ? "" : v);
    }
    for (var i = 2; i < arguments.length; i++) {
      var c = arguments[i];
      if (c == null) continue;
      if (Array.isArray(c)) c.forEach(function (x) { if (x != null) e.appendChild(typeof x === "string" ? document.createTextNode(x) : x); });
      else e.appendChild(typeof c === "string" ? document.createTextNode(c) : c);
    }
    return e;
  }
  var SVGNS = "http://www.w3.org/2000/svg";
  function s(tag, attrs, text) {
    var e = document.createElementNS(SVGNS, tag);
    for (var k in attrs || {}) {
      var v = attrs[k];
      // CSS-variabler settes som stil, sa grafene folger temaet uten ny tegning
      if ((k === "fill" || k === "stroke") && String(v).indexOf("var(") === 0) e.style.setProperty(k, v);
      else e.setAttribute(k, v);
    }
    if (text != null) e.textContent = text;
    return e;
  }
  function fmtDato(iso) { return iso ? iso.slice(8, 10) + "." + iso.slice(5, 7) + "." + iso.slice(0, 4) : ""; }
  function vis(col, v) {
    if (v == null || v === "") return "";
    if (col.type === "date") return fmtDato(v);
    if (col.type === "num") return typeof v === "number" ? nf.format(v) : String(v);
    return String(v);
  }
  function toneAv(tekst) {
    tekst = String(tekst || "");
    if (/^OK/i.test(tekst)) return "ok";
    if (/^Avvik/i.test(tekst)) return "bad";
    if (/^Varsel/i.test(tekst)) return "warn";
    return "neutral";
  }
  function radTone(col, rad) { return col.toneKey ? (rad[col.toneKey] || "neutral") : toneAv(rad[col.key]); }

  var tip = document.getElementById("tip");
  function visTip(ev, linjer) {
    tip.textContent = "";
    linjer.forEach(function (l, i) {
      if (i) tip.appendChild(el("br"));
      if (i === 0) tip.appendChild(el("b", { text: l })); else tip.appendChild(document.createTextNode(l));
    });
    var x = ev.clientX + 14, y = ev.clientY + 14;
    tip.classList.add("show");
    var r = tip.getBoundingClientRect();
    if (x + r.width > window.innerWidth - 8) x = ev.clientX - r.width - 14;
    if (y + r.height > window.innerHeight - 8) y = ev.clientY - r.height - 14;
    tip.style.left = x + "px"; tip.style.top = y + "px";
  }
  function skjulTip() { tip.classList.remove("show"); }

  // ---------- Tema ----------
  (function () {
    var lagret = null;
    try { lagret = localStorage.getItem("ofv-tema"); } catch (e) {}
    if (lagret) document.documentElement.setAttribute("data-theme", lagret);
    document.getElementById("btn-theme").addEventListener("click", function () {
      var na = document.documentElement.getAttribute("data-theme");
      var mork = na ? na === "dark" : window.matchMedia("(prefers-color-scheme: dark)").matches;
      var ny = mork ? "light" : "dark";
      document.documentElement.setAttribute("data-theme", ny);
      try { localStorage.setItem("ofv-tema", ny); } catch (e) {}
      tegnSynligeGrafer();
    });
    document.getElementById("btn-print").addEventListener("click", function () { window.print(); });
  })();

  // ==========================================================
  // Filtrerbar tabell
  // ==========================================================
  function Tabell(spec) {
    var self = this;
    this.spec = spec;
    this.cols = spec.kolonner;
    this.rows = spec.rader;
    this.statusCol = this.cols.filter(function (c) { return c.type === "status"; })[0] || null;
    this.state = { q: "", tone: "alle", f: {}, sort: null, skjult: {}, limit: SIDE };
    this.cols.forEach(function (c) {
      if (c.type === "text" || c.type === "status") {
        var sett = {};
        self.rows.forEach(function (r) { sett[r[c.key] == null ? "" : String(r[c.key])] = 1; });
        c._verdier = Object.keys(sett).sort(coll.compare);
        c._velg = c._verdier.length > 1 && c._verdier.length <= 30;
      }
    });
    this.root = this.bygg();
  }

  Tabell.prototype.bygg = function () {
    var self = this, spec = this.spec;
    var kort = el("div", { cls: "tablecard" });

    // Verktoylinje
    var sok = el("input", { type: "search", placeholder: "Søk i alle kolonner ...", "aria-label": "Søk i tabellen",
      oninput: function () { self.state.q = this.value.trim().toLowerCase(); self.state.limit = SIDE; self.tegn(); } });
    this.sokInput = sok;
    var bar = el("div", { cls: "toolbar" },
      el("label", { cls: "search" }, svgIkon(), sok));
    if (this.statusCol) {
      this.chips = el("div", { cls: "chips", role: "group", "aria-label": "Filtrer på status" });
      bar.appendChild(this.chips);
    }
    bar.appendChild(el("span", { cls: "grow" }));

    var meny = el("div", { cls: "menu" });
    this.cols.forEach(function (c) {
      var cb = el("input", { type: "checkbox", checked: true, onchange: function () {
        if (this.checked) delete self.state.skjult[c.key]; else self.state.skjult[c.key] = 1;
        self.byggHode(); self.tegn();
      } });
      meny.appendChild(el("label", null, cb, c.label));
    });
    bar.appendChild(el("details", { cls: "colmenu" }, el("summary", { cls: "btn" }, "Kolonner"), meny));
    bar.appendChild(el("button", { cls: "btn", type: "button", text: "Nullstill filtre", onclick: function () { self.nullstill(); } }));
    bar.appendChild(el("button", { cls: "btn", type: "button", text: "Last ned CSV", title: "Laster ned radene som vises nå (etter filtrering)", onclick: function () { self.csv(); } }));
    kort.appendChild(bar);

    this.table = el("table");
    this.thead = el("thead");
    this.tbody = el("tbody");
    this.table.appendChild(this.thead);
    this.table.appendChild(this.tbody);
    this.wrap = el("div", { cls: "tablewrap" }, this.table);
    kort.appendChild(this.wrap);
    this.info = el("span");
    this.mer = el("button", { cls: "btn", type: "button", text: "Vis flere", onclick: function () { self.state.limit += SIDE * 5; self.tegn(); } });
    this.alle = el("button", { cls: "btn", type: "button", text: "Vis alle", onclick: function () { self.state.limit = Infinity; self.tegn(); } });
    kort.appendChild(el("div", { cls: "tablefoot" }, this.info, el("span", { cls: "grow", style: "flex:1" }), this.mer, this.alle));
    this.byggHode();
    this.tegn();
    return kort;
  };

  function svgIkon() {
    var g = s("svg", { width: 14, height: 14, viewBox: "0 0 24 24", fill: "none", stroke: "currentColor", "stroke-width": 2.2, "stroke-linecap": "round" });
    g.appendChild(s("circle", { cx: 11, cy: 11, r: 7 }));
    g.appendChild(s("line", { x1: 16.5, y1: 16.5, x2: 21, y2: 21 }));
    return g;
  }

  Tabell.prototype.synlige = function () {
    var sk = this.state.skjult;
    return this.cols.filter(function (c) { return !sk[c.key]; });
  };

  Tabell.prototype.byggHode = function () {
    var self = this;
    this.thead.textContent = "";
    var tr1 = el("tr", { cls: "labels" }), tr2 = el("tr", { cls: "filters" });
    this.synlige().forEach(function (c) {
      var sortert = self.state.sort && self.state.sort.key === c.key;
      var pil = sortert ? (self.state.sort.dir > 0 ? "▲" : "▼") : "⇅";
      tr1.appendChild(el("th", { cls: sortert ? "sorted" : null, title: "Klikk for å sortere",
        "aria-sort": sortert ? (self.state.sort.dir > 0 ? "ascending" : "descending") : null,
        onclick: function () { self.sorter(c.key); } }, c.label, el("span", { cls: "arrow", text: pil })));
      tr2.appendChild(el("th", null, self.filterFelt(c)));
    });
    this.thead.appendChild(tr1);
    this.thead.appendChild(tr2);
  };

  Tabell.prototype.filterFelt = function (c) {
    var self = this, f = this.state.f[c.key] || {};
    function sett(k, v) {
      var cur = self.state.f[c.key] || {};
      cur[k] = v;
      self.state.f[c.key] = cur;
      self.state.limit = SIDE;
      self.tegn();
    }
    function aktiv(inp, v) { inp.classList.toggle("active", v != null && v !== ""); }
    if (c.type === "num" || c.type === "date") {
      var typ = c.type === "num" ? "number" : "date";
      var a = el("input", { type: typ, placeholder: "min", title: c.type === "date" ? "Fra og med" : "Minimum", value: f.min || null,
        oninput: function () { aktiv(this, this.value); sett("min", this.value); } });
      var b = el("input", { type: typ, placeholder: "maks", title: c.type === "date" ? "Til og med" : "Maksimum", value: f.max || null,
        oninput: function () { aktiv(this, this.value); sett("max", this.value); } });
      aktiv(a, f.min); aktiv(b, f.max);
      return el("div", { cls: "range" }, a, b);
    }
    if (c._velg) {
      var sel = el("select", { "aria-label": "Filtrer " + c.label, onchange: function () { aktiv(this, this.value); sett("eq", this.value); } },
        el("option", { value: "", text: "Alle" }));
      c._verdier.forEach(function (v) {
        sel.appendChild(el("option", { value: "=" + v, text: v === "" ? "(tom)" : v, selected: f.eq === "=" + v }));
      });
      aktiv(sel, f.eq);
      return sel;
    }
    var inp = el("input", { type: "text", placeholder: "Filtrer ...", value: f.like || null, "aria-label": "Filtrer " + c.label,
      oninput: function () { aktiv(this, this.value); sett("like", this.value.trim().toLowerCase()); } });
    aktiv(inp, f.like);
    return inp;
  };

  Tabell.prototype.sorter = function (key) {
    var st = this.state.sort;
    if (!st || st.key !== key) this.state.sort = { key: key, dir: 1 };
    else if (st.dir === 1) st.dir = -1;
    else this.state.sort = null;
    this.byggHode();
    this.tegn();
  };

  Tabell.prototype.nullstill = function () {
    this.state.q = ""; this.state.tone = "alle"; this.state.f = {}; this.state.sort = null; this.state.limit = SIDE;
    this.sokInput.value = "";
    this.byggHode();
    this.tegn();
  };

  Tabell.prototype.settTone = function (tone) {
    this.state.tone = tone; this.state.limit = SIDE; this.tegn();
  };

  Tabell.prototype.passerUtenTone = function (r) {
    var cols = this.cols, f = this.state.f;
    for (var i = 0; i < cols.length; i++) {
      var c = cols[i], ff = f[c.key];
      if (!ff) continue;
      var v = r[c.key];
      if (ff.eq) { if ((v == null ? "" : String(v)) !== ff.eq.slice(1)) return false; }
      if (ff.like) { if (vis(c, v).toLowerCase().indexOf(ff.like) < 0) return false; }
      if (ff.min) {
        if (v == null || v === "") return false;
        if (c.type === "num" ? v < parseFloat(ff.min) : v < ff.min) return false;
      }
      if (ff.max) {
        if (v == null || v === "") return false;
        if (c.type === "num" ? v > parseFloat(ff.max) : v > ff.max) return false;
      }
    }
    if (this.state.q) {
      var syn = this.synlige(), hay = "";
      for (var j = 0; j < syn.length; j++) hay += " " + vis(syn[j], r[syn[j].key]);
      hay = hay.toLowerCase();
      var ord = this.state.q.split(/\s+/);
      for (var k = 0; k < ord.length; k++) if (hay.indexOf(ord[k]) < 0) return false;
    }
    return true;
  };

  Tabell.prototype.filtrert = function () {
    var self = this, sc = this.statusCol, tone = this.state.tone;
    var base = this.rows.filter(function (r) { return self.passerUtenTone(r); });
    this._toneTelling = { alle: base.length, ok: 0, warn: 0, bad: 0, neutral: 0 };
    if (sc) base.forEach(function (r) { self._toneTelling[toneAv(r[sc.key])]++; });
    var ut = tone === "alle" || !sc ? base : base.filter(function (r) { return toneAv(r[sc.key]) === tone; });
    var st = this.state.sort;
    if (st) {
      var col = this.cols.filter(function (c) { return c.key === st.key; })[0];
      ut = ut.slice().sort(function (a, b) {
        var x = a[st.key], y = b[st.key];
        var xe = x == null || x === "", ye = y == null || y === "";
        if (xe && ye) return 0;
        if (xe) return 1;
        if (ye) return -1;
        var r = col.type === "num" ? x - y : coll.compare(String(x), String(y));
        return r * st.dir;
      });
    }
    return ut;
  };

  Tabell.prototype.tegnChips = function () {
    if (!this.chips) return;
    var self = this, t = this._toneTelling;
    this.chips.textContent = "";
    ["alle", "ok", "warn", "bad", "neutral"].forEach(function (tone) {
      if (tone !== "alle" && !t[tone] && self.state.tone !== tone) return;
      var b = el("button", { cls: "chip", type: "button", "aria-pressed": String(self.state.tone === tone),
        onclick: function () { self.settTone(tone); } },
        tone === "alle" ? null : el("span", { cls: "dot " + tone }),
        tone === "alle" ? "Alle" : TONE_NAVN[tone],
        el("span", { cls: "n", text: nf.format(t[tone]) }));
      self.chips.appendChild(b);
    });
  };

  Tabell.prototype.tegn = function () {
    var self = this, rader = this.filtrert(), syn = this.synlige();
    this._sist = rader;
    this.tegnChips();
    this.tbody.textContent = "";
    var frag = document.createDocumentFragment();
    var n = Math.min(rader.length, this.state.limit);
    for (var i = 0; i < n; i++) {
      var r = rader[i], tr = document.createElement("tr");
      for (var j = 0; j < syn.length; j++) {
        var c = syn[j], v = r[c.key], txt = vis(c, v);
        var td = document.createElement("td");
        td.className = "t-" + c.type + (txt === "" ? " empty" : "");
        if (txt !== "" && (c.type === "status" || c.toneKey)) {
          td.appendChild(el("span", { cls: "pill " + radTone(c, r), text: txt }));
          td.title = txt;
        } else {
          td.textContent = txt === "" ? "–" : txt;
          if (txt.length > 40) td.title = txt;
        }
        tr.appendChild(td);
      }
      frag.appendChild(tr);
    }
    if (!rader.length) {
      frag.appendChild(el("tr", null, el("td", { colspan: syn.length, cls: "noresult", text: "Ingen rader matcher filtrene." })));
    }
    this.tbody.appendChild(frag);
    var tot = this.rows.length;
    this.info.textContent = "Viser " + nf.format(n) + (n < rader.length ? " av " + nf.format(rader.length) + " treff" : "") +
      (rader.length !== tot ? " (filtrert fra " + nf.format(tot) + " rader)" : n === tot ? " rader" : "");
    this.mer.style.display = n < rader.length ? "" : "none";
    this.alle.style.display = n < rader.length ? "" : "none";
  };

  Tabell.prototype.csv = function () {
    var syn = this.synlige(), rader = this._sist || [];
    function q(x) { x = String(x == null ? "" : x); return /[";\n\r]/.test(x) ? '"' + x.replace(/"/g, '""') + '"' : x; }
    var linjer = [syn.map(function (c) { return q(c.label); }).join(";")];
    rader.forEach(function (r) { linjer.push(syn.map(function (c) { return q(vis(c, r[c.key])); }).join(";")); });
    var blob = new Blob(["\uFEFF" + linjer.join("\r\n")], { type: "text/csv;charset=utf-8" });
    var a = el("a", { href: URL.createObjectURL(blob), download: (D.selskap + " - " + this.spec.tittel).replace(/[\\/:*?"<>|]/g, "_") + ".csv" });
    document.body.appendChild(a); a.click(); a.remove();
    setTimeout(function () { URL.revokeObjectURL(a.href); }, 1000);
  };

  // Tabellkort (med tittel), evt. sammenleggbart og lat-tegnet
  function tabellKort(spec, lukket) {
    var hode = el("div", { cls: "card-head" },
      lukket ? el("span", { cls: "chev", text: "▶" }) : null,
      el("div", null, el("h3", { text: spec.tittel + "  " }, el("span", { cls: "num", style: "color:var(--muted);font-weight:500", text: "(" + nf.format(spec.rader.length) + ")" })),
        spec.beskrivelse ? el("p", { text: spec.beskrivelse }) : null));
    if (!lukket) {
      var t = new Tabell(spec);
      var kort = el("div", { cls: "card", id: "t-" + spec.id }, hode, t.root);
      kort._tabell = t;
      return kort;
    }
    var d = el("details", { cls: "card", id: "t-" + spec.id }, el("summary", null, hode));
    d.addEventListener("toggle", function () {
      if (d.open && !d._tabell) { d._tabell = new Tabell(spec); d.appendChild(d._tabell.root); }
    });
    return d;
  }

  // ==========================================================
  // Grafer (ren SVG)
  // ==========================================================
  function css(navn) { return "var(" + navn + ")"; }
  function toneFarge(t) { return css({ ok: "--ok", warn: "--warn", bad: "--bad", neutral: "--neutral" }[t] || "--series"); }
  // Pen akse-maks for antall: 4 like steg med heltall (1, 2, 5, 10, 20, 50 ...)
  function pent(maks) {
    var steg = Math.max(1, Math.ceil(maks / 4));
    var p = Math.pow(10, Math.floor(Math.log10(steg))), m = steg / p;
    return (m <= 1 ? 1 : m <= 2 ? 2 : m <= 5 ? 5 : 10) * p * 4;
  }
  function tekstBredde(t, px) { return String(t).length * (px || 12) * 0.56; }

  function hbar(g, svgEl, B) {
    var data = g.data, sum = data.reduce(function (a, d) { return a + d.value; }, 0);
    var maks = Math.max.apply(null, data.map(function (d) { return d.value; }).concat([1]));
    var lw = Math.min(B * 0.42, Math.max.apply(null, data.map(function (d) { return tekstBredde(d.label, 12.5); })) + 12);
    var vw = 64, bw = Math.max(40, B - lw - vw), rh = 28, H = data.length * rh + 4;
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    data.forEach(function (d, i) {
      var y = i * rh, w = Math.max(d.value > 0 ? 3 : 0, d.value / maks * bw);
      var lab = d.label;
      while (tekstBredde(lab, 12.5) > lw - 10 && lab.length > 4) lab = lab.slice(0, -2);
      if (lab !== d.label) lab = lab.replace(/.$/, "…");
      svgEl.appendChild(s("text", { x: lw - 10, y: y + rh / 2 + 4, "text-anchor": "end", "font-size": 12.5, fill: css("--text") }, lab));
      svgEl.appendChild(s("rect", { x: lw, y: y + 7, width: w, height: rh - 14, rx: 3, fill: d.tone ? toneFarge(d.tone) : d.muted ? css("--series-muted") : css("--series") }));
      svgEl.appendChild(s("text", { x: lw + w + 8, y: y + rh / 2 + 4, "font-size": 12, fill: css("--text-2"), class: "num" }, nf.format(d.value)));
      var hit = s("rect", { x: 0, y: y, width: B, height: rh, fill: "transparent" });
      hit.addEventListener("mousemove", function (ev) { visTip(ev, [d.label, nf.format(d.value) + " (" + pf.format(sum ? d.value / sum : 0) + ")"]); });
      hit.addEventListener("mouseleave", skjulTip);
      svgEl.appendChild(hit);
    });
  }

  function columns(g, svgEl, B) {
    var data = g.data, H = 230, top = 12, bunn = 26, venstre = 36;
    var maks = pent(Math.max.apply(null, data.map(function (d) { return d.value; }).concat([1])));
    var ih = H - top - bunn, iw = B - venstre - 4, steg = iw / data.length;
    var bw = Math.max(4, Math.min(48, steg * 0.62));
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    for (var k = 0; k <= 4; k++) {
      var v = maks * k / 4, y = top + ih - ih * k / 4;
      svgEl.appendChild(s("line", { x1: venstre, x2: B, y1: y, y2: y, stroke: k ? css("--grid") : css("--border-strong"), "stroke-width": 1 }));
      svgEl.appendChild(s("text", { x: venstre - 6, y: y + 4, "text-anchor": "end", "font-size": 11, fill: css("--muted") }, nf.format(v)));
    }
    var hvert = Math.ceil(data.length / Math.max(1, Math.floor(iw / 46)));
    data.forEach(function (d, i) {
      var x = venstre + i * steg + (steg - bw) / 2, h = d.value / maks * ih;
      if (d.value > 0) svgEl.appendChild(s("path", { d: rundTopp(x, top + ih - h, bw, h, Math.min(4, bw / 2, h)), fill: d.tone ? toneFarge(d.tone) : css("--series") }));
      if (i % hvert === 0 || data.length <= 12)
        svgEl.appendChild(s("text", { x: x + bw / 2, y: H - 8, "text-anchor": "middle", "font-size": 11, fill: css("--text-2") }, d.label));
      var hit = s("rect", { x: venstre + i * steg, y: top, width: steg, height: ih, fill: "transparent" });
      hit.addEventListener("mousemove", function (ev) { visTip(ev, [d.label, nf.format(d.value)]); });
      hit.addEventListener("mouseleave", skjulTip);
      svgEl.appendChild(hit);
    });
    // Direkte etikett pa hoyeste soyle
    var mi = 0; data.forEach(function (d, i) { if (d.value > data[mi].value) mi = i; });
    if (data[mi] && data[mi].value > 0) {
      var xm = venstre + mi * steg + steg / 2, ym = top + ih - data[mi].value / maks * ih - 6;
      svgEl.insertBefore(s("text", { x: xm, y: Math.max(10, ym), "text-anchor": "middle", "font-size": 11.5, "font-weight": 600, fill: css("--text") }, nf.format(data[mi].value)), svgEl.firstChild);
    }
  }
  function rundTopp(x, y, w, h, r) {
    return "M" + x + "," + (y + h) + " V" + (y + r) + " Q" + x + "," + y + " " + (x + r) + "," + y +
      " H" + (x + w - r) + " Q" + (x + w) + "," + y + " " + (x + w) + "," + (y + r) + " V" + (y + h) + " Z";
  }

  function stack(g, svgEl, B) {
    var rh = 54, H = g.rows.length * rh;
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    g.rows.forEach(function (row, i) {
      var y = i * rh, sum = row.segments.reduce(function (a, d) { return a + d.value; }, 0), x = 0;
      svgEl.appendChild(s("text", { x: 0, y: y + 14, "font-size": 12.5, "font-weight": 600, fill: css("--text") }, row.label));
      svgEl.appendChild(s("text", { x: B, y: y + 14, "text-anchor": "end", "font-size": 12, fill: css("--text-2") }, nf.format(sum) + " biler"));
      if (!sum) return;
      row.segments.forEach(function (seg) {
        if (!seg.value) return;
        var w = seg.value / sum * B, gap = 2;
        var r = s("rect", { x: x, y: y + 22, width: Math.max(1, w - gap), height: 18, rx: 3, fill: toneFarge(seg.tone) });
        r.addEventListener("mousemove", function (ev) { visTip(ev, [row.label + " – " + seg.label, nf.format(seg.value) + " av " + nf.format(sum) + " (" + pf.format(seg.value / sum) + ")"]); });
        r.addEventListener("mouseleave", skjulTip);
        svgEl.appendChild(r);
        var lbl = pf.format(seg.value / sum);
        if (w > tekstBredde(lbl, 11.5) + 12) {
          var t = s("text", { x: x + 7, y: y + 35, "font-size": 11.5, "font-weight": 600, fill: "#ffffff", "pointer-events": "none" }, lbl);
          svgEl.appendChild(t);
        }
        x += w;
      });
    });
  }

  var TEGNERE = { hbar: hbar, columns: columns, stack: stack };
  var grafKort = [];

  function grafKortFor(g) {
    var svgEl = s("svg", { role: "img", "aria-label": g.tittel });
    var body = el("div", { cls: "chartbody" }, svgEl);
    var kort = el("div", { cls: "card" + (g.bred ? " wide" : "") },
      el("div", { cls: "card-head" }, el("h3", { text: g.tittel }), g.undertittel ? el("p", { text: g.undertittel }) : null),
      body);
    // Tegnforklaring
    var leg = null;
    if (g.type === "stack") {
      var sett = {};
      g.rows.forEach(function (r) { r.segments.forEach(function (sg) { if (sg.value) sett[sg.label + "|" + sg.tone] = 1; }); });
      leg = Object.keys(sett).map(function (k) { var p = k.split("|"); return [p[0], p[1]]; });
    } else if (g.statusLegend) {
      leg = [["OK", "ok"], ["Varsel", "warn"], ["Avvik", "bad"]];
    }
    if (leg) kort.insertBefore(el("div", { cls: "legend" }, leg.map(function (l) {
      return el("span", null, el("i", { style: "background:var(--" + l[1] + ")" }), l[0]);
    })), body);
    // Datatabell (tilgjengelighet / eksakte tall)
    var tb = el("tbody");
    if (g.type === "stack") g.rows.forEach(function (r) { r.segments.forEach(function (sg) {
      tb.appendChild(el("tr", null, el("td", { text: r.label }), el("td", { text: sg.label }), el("td", { cls: "n", text: nf.format(sg.value) })));
    }); });
    else g.data.forEach(function (d) { tb.appendChild(el("tr", null, el("td", { text: d.label }), el("td", { cls: "n", text: nf.format(d.value) }))); });
    kort.appendChild(el("details", { cls: "datatable" }, el("summary", { text: "Vis tallene" }), el("table", null, tb)));
    kort._tegn = function () {
      var B = Math.max(260, body.clientWidth - 36);
      if (!body.clientWidth) return;
      svgEl.textContent = "";
      TEGNERE[g.type](g, svgEl, B);
    };
    grafKort.push(kort);
    return kort;
  }
  function tegnSynligeGrafer() { grafKort.forEach(function (k) { if (k.offsetParent) k._tegn(); }); }
  var rt;
  window.addEventListener("resize", function () { clearTimeout(rt); rt = setTimeout(tegnSynligeGrafer, 120); });

  // ==========================================================
  // Sider
  // ==========================================================
  var main = document.getElementById("main"), tabs = document.getElementById("tabs");
  var paneler = {};
  document.getElementById("hdr-company").textContent = D.selskap + " · generert " + D.generert;

  function kpiFliser(kpier, tabellKortEl) {
    var wrap = el("div", { cls: "kpis" });
    kpier.forEach(function (k) {
      var klikkbar = !!(k.tone && tabellKortEl);
      var tile = el(klikkbar ? "button" : "div", { cls: "kpi", type: klikkbar ? "button" : null,
        title: klikkbar ? "Vis bare disse radene i tabellen under" : null },
        el("div", { cls: "label" }, k.tone ? el("span", { cls: "dot " + k.tone }) : null, k.label),
        el("div", { cls: "value num", text: nf.format(k.verdi) }));
      if (klikkbar) tile.addEventListener("click", function () {
        var t = tabellKortEl._tabell, ny = t.state.tone === k.tone ? "alle" : k.tone;
        t.settTone(ny);
        wrap.querySelectorAll(".kpi").forEach(function (x) { x.classList.remove("active"); });
        if (ny !== "alle") tile.classList.add("active");
        tabellKortEl.scrollIntoView({ behavior: "smooth", block: "start" });
      });
      wrap.appendChild(tile);
    });
    return wrap;
  }

  function metaChips(meta) {
    return el("div", { cls: "meta" }, meta.map(function (m) {
      return el("span", { title: m.label + ": " + m.verdi }, m.label + ": ", el("b", { text: String(m.verdi) }));
    }));
  }

  function antallAvvik(f) {
    var n = 0;
    f.tabeller.forEach(function (t) {
      if (/historikk$/.test(t.id)) return;
      var sc = t.kolonner.filter(function (c) { return c.type === "status"; })[0];
      if (sc) t.rader.forEach(function (r) { if (toneAv(r[sc.key]) === "bad") n++; });
    });
    return n;
  }

  function leggTilFane(id, tittel, badge, bygg) {
    var knapp = el("button", { cls: "tab", role: "tab", type: "button", "aria-selected": "false", "aria-controls": "p-" + id, id: "tab-" + id,
      onclick: function () { velg(id); } }, tittel);
    if (badge != null) knapp.appendChild(el("span", { cls: "badge" + (badge ? "" : " zero"), title: "Antall avvik", text: nf.format(badge) }));
    tabs.appendChild(knapp);
    var panel = el("section", { cls: "panel", id: "p-" + id, role: "tabpanel", "aria-labelledby": "tab-" + id });
    main.appendChild(panel);
    paneler[id] = { knapp: knapp, panel: panel, bygg: bygg, bygget: false };
  }

  function velg(id) {
    if (!paneler[id]) id = "oversikt";
    Object.keys(paneler).forEach(function (k) {
      var p = paneler[k], aktiv = k === id;
      p.knapp.setAttribute("aria-selected", String(aktiv));
      p.panel.classList.toggle("active", aktiv);
      if (aktiv && !p.bygget) { p.bygg(p.panel); p.bygget = true; }
    });
    if (location.hash !== "#" + id) history.replaceState(null, "", "#" + id);
    window.scrollTo(0, 0);
    requestAnimationFrame(tegnSynligeGrafer);
  }

  // --- Oversikt ---
  leggTilFane("oversikt", "Oversikt", null, function (p) {
    p.appendChild(el("div", { cls: "pagehead" },
      el("h2", { text: "Oversikt" }),
      el("p", { text: "Samlet resultat for OFV-kontrollene. Klikk deg inn på en kontroll for detaljer, filtrering og eksport. " +
        "Analyse-fanen viser nøkkeltall og grafer på tvers av kontrollene." })));
    p.appendChild(el("div", { cls: "runinfo" },
      el("span", null, "Generert: ", el("b", { text: D.generert })),
      el("span", null, "Tidsbruk: ", el("b", { text: nf.format(D.tidsbruk) + " s" })),
      el("span", null, "OFV-oppslag: ", el("b", { text: nf.format(D.api.ofvKjoretoy) + " kjøretøy" }),
        " (" + nf.format(D.api.ofvBuffer) + " gjenbrukt, " + nf.format(D.api.ofvHttp) + " HTTP-kall)"),
      el("span", null, "SVV-kall: ", el("b", { text: nf.format(D.api.svvHttp) }))));
    var grid = el("div", { cls: "overview-grid" });
    D.oversikt.forEach(function (o) {
      grid.appendChild(el("div", { cls: "card ov" },
        el("h3", { text: o.tittel }),
        metaChips(o.meta.filter(function (m) { return m.label !== "Kildefil"; })),
        el("div", { cls: "minis" }, o.kpier.map(function (k) {
          return el("div", { cls: "mini" },
            el("div", { cls: "l" }, k.tone ? el("span", { cls: "dot " + k.tone }) : null, k.label),
            el("div", { cls: "v num", text: nf.format(k.verdi) }));
        })),
        el("button", { cls: "open", type: "button", text: "Åpne kontrollen →", onclick: function () { velg(o.id); } })));
    });
    p.appendChild(grid);
    p.appendChild(tabellKort(D.input, true));
  });

  // --- En fane per kontroll ---
  D.faner.forEach(function (f) {
    leggTilFane(f.id, f.tittel, antallAvvik(f), function (p) {
      var kort = f.tabeller.map(function (t) { return tabellKort(t, /historikk$/.test(t.id)); });
      p.appendChild(el("div", { cls: "pagehead" },
        el("h2", { text: f.fullTittel }),
        metaChips(f.meta),
        el("details", { cls: "rule", open: true }, el("summary", { text: "Kontrollregel" }), el("p", { text: f.regel }))));
      p.appendChild(kpiFliser(f.kpier, kort[0]));
      kort.forEach(function (k) { p.appendChild(k); });
    });
  });

  // --- Analyse ---
  leggTilFane("analyse", "Analyse", null, function (p) {
    p.appendChild(el("div", { cls: "pagehead" },
      el("h2", { text: "Analyse" }),
      el("p", { text: "Nøkkeltall og fordelinger på tvers av kontrollene. Hold musen over en graf for eksakte tall, eller åpne «Vis tallene» under hver graf." })));
    p.appendChild(kpiFliser(D.analyse.nokkeltall, null));
    var grid = el("div", { cls: "chartgrid" });
    D.analyse.grafer.forEach(function (g) { grid.appendChild(grafKortFor(g)); });
    p.appendChild(grid);
  });

  document.getElementById("footer").textContent =
    "Kilder: OFV Transactions API og Statens vegvesen (reserve). Rapporten er generert av 02_OFV_API_Kontroller i Databricks.";

  window.addEventListener("hashchange", function () { velg(location.hash.slice(1)); });
  window.addEventListener("beforeprint", function () {
    Object.keys(paneler).forEach(function (k) { var p = paneler[k]; if (!p.bygget) { p.bygg(p.panel); p.bygget = true; } });
  });
  velg(location.hash.slice(1) || "oversikt");
})();
</script>
</body>
</html>
"""


skriv_html_rapport(_bundles, OUTPUT_HTML)
print(f"HTML-rapport lagret: {OUTPUT_HTML}")

In [ ]:
# ==================================================================
# CELLE 6: OPPSUMMERING
# ==================================================================

print("=" * 70)
print("OFV-KONTROLLER - OPPSUMMERING")
print("=" * 70)
print(f"Selskap: {SELSKAP}")
print(f"Tidsbruk: {_elapsed_s:.1f} sekunder")
print(f"OFV: {API_TELLER['ofv_kjoretoy']} kjoretoy slatt opp, "
      f"{API_TELLER['ofv_buffer']} gjenbrukt fra buffer, "
      f"{API_TELLER['ofv_http']} HTTP-kall. SVV: {API_TELLER['svv_http']} HTTP-kall.\n")

for _navn, _kjort in KJOR_KONTROLL.items():
    if not _kjort:
        print(f"--- {_navn}: HOPPET OVER (inputfil mangler) ---\n")

for bundle in _bundles:
    print(f"--- {bundle['kontroll']} ---")
    print(f"  Kildefil: {bundle['kildefil']}")
    print(f"  Orgnr: {bundle['orgnr'] or '(ikke oppgitt)'}")
    if bundle["dato_fra"] and bundle["dato_til"]:
        print(f"  Periode: {bundle['dato_fra']:%d.%m.%Y} - {bundle['dato_til']:%d.%m.%Y}")
    if bundle["kontroll"] in TIDSBRUK_PER_KONTROLL:
        print(f"  Tidsbruk: {TIDSBRUK_PER_KONTROLL[bundle['kontroll']]:.1f} s")
    for key, value in bundle["stats"].items():
        print(f"  {key}: {value}")
    print()

print(f"Excel-arbeidsbok: {OUTPUT_XLSX}")
print(f"HTML-rapport: {OUTPUT_HTML}")